In [ ]:
# 버전 고정 — 실행 후 [런타임 > 세션 다시 시작], 그다음 S1·S2·S2-1 재실행
!pip -q install "transformers==5.17.0" "accelerate==1.14.0"

In [ ]:
import os, sys, json, time
from google.colab import drive
drive.mount("/content/drive")

MD   = "/content/drive/MyDrive/"
ROOT = MD + "11주차_A축_실험_다영/"
HK   = MD + "SOTA_다영/hk/"
PKG  = HK + "pkg/"
OUT  = HK + "results_p8s/"

os.environ["ABENCH_ROOT"] = ROOT
os.environ["ABENCH_P9"]   = ROOT + "P9/"
os.environ["ABENCH_A0"]   = ROOT + "A0_headroom/"
for d in (PKG, OUT):
    os.makedirs(d, exist_ok=True)
assert not PKG.startswith(os.environ["ABENCH_P9"]), "공유 폴더에 쓰지 않는다"
assert not OUT.startswith(os.environ["ABENCH_P9"]), "공유 폴더에 쓰지 않는다"

need = {
    "P9 프롬프트 모듈":   "P9/steer_v3/code/p8s_prompt.py",
    "P9 w_v3":           "P9/steer_v3/w_v3.npz",
    "P9 라벨":            "P9/steer_v3/labels_v3.parquet",
    "P9 cb_v3":          "P9/steer_v3/cb_v3.parquet",
    "P9 manifest":       "P9/steer_v3/env/row_manifest.parquet",
    "P9 env json":       "P9/steer_v3/env/manifest_cell1.json",
    "P9 hidden":         "P9/hidden_v2/p8s_v2_h_out.npy",
    "P9 behavior":       "P9/hidden_v2/p8s_behavior_v2.csv",
    "A0 base":           "A0_headroom/gen_base.parquet",
    "A0 late_ret_r0.06": "A0_headroom/gen_late_ret_r0.06.parquet",
    "A0 late_ret_r0.09": "A0_headroom/gen_late_ret_r0.09.parquet",
}
miss = [k for k, v in need.items() if not os.path.exists(ROOT + v)]
for k, v in need.items():
    print(("OK   " if os.path.exists(ROOT + v) else "없음 ") + f"{k:20s} {v}")
assert not miss, f"없는 산출물: {miss}  (S1-1 로 실제 위치를 찾는다)"
print(f"\n코드 {PKG}\n결과 {OUT}")

In [ ]:
!pip -q install pyarrow 2>/dev/null
print("준비 완료")

In [ ]:
# 모듈 소스 (base64). 줄바꿈 손상에 안전하다.
import base64, pathlib

FORCE_COMMON = False

FILES = {}
FILES["common/artifacts.py"] = "IiIi7IKw7Lac66y8IOuhnOuNlC4KCuydtCDtlZjrhKTsiqTripQg6riw7KSA7ISg7J2EIOyerOq1rO2YhO2VmOyngCDslYrripTri6QuIEEwL0ExIOyXkOyEnCDsnbTrr7gg6rKA7Kad65CcIOyCsOy2nOusvOydhCDsnb3ripTri6QuCiAgLSDtmITqsr0g6rO17JygIO2PtOuNlCBQOS8gICjsnb3quLAg7KCE7JqpKSA6IHdfdjMsIO2UhOuhrO2UhO2KuCDrqqjrk4gsIOudvOuyqCwgZXhwMiDrjIDsobDrs7gKICAtIOuLpOyYgSBBMF9oZWFkcm9vbS8gICAgICAgICAgICAgIDogZ2VuXyoucGFycXVldCAo7ZaJIOuLqOychCByYXcvcHJlZC9lbSkKCuyerOq1rO2YhO2VmOuptCBBMC00IOydmCDsnqztmIQg6rKA7KadKOusuOyekOyXtCDrtojsnbzsuZggPD0yMC8xMjI0KeydhCDri6Tsi5wg7Ya16rO87ZW07JW8IO2VmOuKlOuNsCwK6re465+0IOydtOycoOqwgCDsl4bri6QuIOydtOuvuCDthrXqs7ztlZwg7IKw7Lac66y87J2EIOq3uOuMgOuhnCDsk7Tri6QuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaW1wb3J0bGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCiMgQTAvQTEg64W47Yq467aB6rO8IOuPmeydvO2VnCDquLDrs7gg6rK966GcLiDtmZjqsr3rs4DsiJjroZwg642u7Ja07JO4IOyImCDsnojri6QuClJPT1QgPSBvcy5lbnZpcm9uLmdldCgiQUJFTkNIX1JPT1QiLAogICAgICAgICAgICAgICAgICAgICAgIi9jb250ZW50L2RyaXZlL015RHJpdmUvMTHso7zssKhfQey2lV/si6Ttl5hf64uk7JiBLyIpClA5ID0gb3MuZW52aXJvbi5nZXQoIkFCRU5DSF9QOSIsIFJPT1QgKyAiUDkvIikKU1YzID0gUDkgKyAic3RlZXJfdjMvIgpBMCA9IG9zLmVudmlyb24uZ2V0KCJBQkVOQ0hfQTAiLCBST09UICsgIkEwX2hlYWRyb29tLyIpCgojIEEwLTIg7JmAIOuPmeydvO2VnCBtZDUuIO2UhOuhrO2UhO2KuCDrqqjrk4jsnbQg67CU64CM66m0IOq4sOykgOyEoCDsiJjsuZjqsIAg64us65287KeE64ukLgpQUk9NUFRfTUQ1ID0gIjIwOTU5YTRjMzQyMDNlYTA4NjZkMDMwYmFmMjM1OTVkIgoKIyBBMCDqsIAg7IOd7ISx7ZWcIO2WieuPmSDsp5HtlakuIGxhdGVfcmV0X3IwLjA2IOydtCDrj5nqsrAg7ISk7KCVKD0gYWx3YXlzIHN0ZWVyaW5nKS4KQVJNUyA9IFsKICAgICJiYXNlIiwKICAgICJlYXJseV9yZXRfcjAuMDMiLCAiZWFybHlfcmV0X3IwLjA2IiwgImVhcmx5X2FjY19yMC4wMyIsICJlYXJseV9hY2NfcjAuMDYiLAogICAgImxhdGVfcmV0X3IwLjAzIiwgImxhdGVfcmV0X3IwLjA2IiwgImxhdGVfcmV0X3IwLjA5IiwgImxhdGVfcmV0X3IwLjEyIiwKICAgICJsYXRlX2FjY19yMC4wMyIsICJsYXRlX2FjY19yMC4wNiIsCl0KUkVGX0FSTSA9ICJsYXRlX3JldF9yMC4wNiIKCgpkZWYgX21kNShwYXRoOiBzdHIsIGNodW5rOiBpbnQgPSAxIDw8IDI0KSAtPiBzdHI6CiAgICBpbXBvcnQgaGFzaGxpYgogICAgaCA9IGhhc2hsaWIubWQ1KCkKICAgIHdpdGggb3BlbihwYXRoLCAicmIiKSBhcyBmOgogICAgICAgIGZvciBiIGluIGl0ZXIobGFtYmRhOiBmLnJlYWQoY2h1bmspLCBiIiIpOgogICAgICAgICAgICBoLnVwZGF0ZShiKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpAZGF0YWNsYXNzCmNsYXNzIEJlbmNoOgogICAgbTogcGQuRGF0YUZyYW1lICAgICAgICAgICAgIyBwOHNfYmVoYXZpb3JfdjIgKDEsMjI07ZaJKQogICAgbWFuOiBwZC5EYXRhRnJhbWUgICAgICAgICAgIyByb3dfbWFuaWZlc3QgKGZvbGQsIG5fcHJvbXB0X3RvaywgcHJvbXB0X21kNSkKICAgIGNiOiBwZC5EYXRhRnJhbWUgICAgICAgICAgICMgY2JfdjMgKGNsb3NlZC1ib29rIOuLtSwga3NpZGVfdjMpCiAgICBsYWJlbHM6IHBkLkRhdGFGcmFtZSAgICAgICAjIGxhYmVsc192MyAocmV0X3gsIGFjY194LCBwcm9wKQogICAgUFA6IG9iamVjdCAgICAgICAgICAgICAgICAgIyBwOHNfcHJvbXB0IOuqqOuTiAogICAgZm9sZDogbnAubmRhcnJheQogICAgeUM6IG5wLm5kYXJyYXkgICAgICAgICAgICAgIyAxID0gQysKICAgIHlLOiBucC5uZGFycmF5ICAgICAgICAgICAgICMgMSA9IEsrCiAgICBjZWxsOiBucC5uZGFycmF5ICAgICAgICAgICAjICdLK0MtJyDrk7EKICAgIG1vZGVsX3JldmlzaW9uOiBzdHIKCiAgICBAcHJvcGVydHkKICAgIGRlZiBuKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYubSkKCiAgICBAcHJvcGVydHkKICAgIGRlZiBpc19jbmVnKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgcmV0dXJuIHNlbGYueUMgPT0gMAoKCmRlZiBsb2FkX2JlbmNoKCkgLT4gQmVuY2g6CiAgICBuZWVkID0gWyJoaWRkZW5fdjIvcDhzX2JlaGF2aW9yX3YyLmNzdiIsICJzdGVlcl92My9lbnYvcm93X21hbmlmZXN0LnBhcnF1ZXQiLAogICAgICAgICAgICAic3RlZXJfdjMvZW52L21hbmlmZXN0X2NlbGwxLmpzb24iLCAic3RlZXJfdjMvY29kZS9wOHNfcHJvbXB0LnB5IiwKICAgICAgICAgICAgInN0ZWVyX3YzL2xhYmVsc192My5wYXJxdWV0IiwgInN0ZWVyX3YzL2NiX3YzLnBhcnF1ZXQiXQogICAgZm9yIGYgaW4gbmVlZDoKICAgICAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoUDkgKyBmKToKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgKICAgICAgICAgICAgICAgIGYi7JeG7J2MOiB7UDkgKyBmfVxuIgogICAgICAgICAgICAgICAgIlA5IOqzteycoCDtj7TrjZQg67CU66Gc6rCA6riwKOydveq4sCDsoITsmqkp6rCAIOyXsOqysOuPvCDsnojripTsp4Ag66i87KCAIO2ZleyduO2VnOuLpC4gIgogICAgICAgICAgICAgICAgIuygkeq3vOydtCDrgYrqsrzri6TrqbQg7ZiE6rK97JeQ6rKMIOyerOqzteycoOulvCDsmpTssq3tlbTslbwg7ZWc64ukLiIpCgogICAgc3lzLnBhdGguaW5zZXJ0KDAsIFNWMyArICJjb2RlIikKICAgIGltcG9ydCBwOHNfcHJvbXB0IGFzIFBQICAjIG5vcWE6IE44MTMKICAgIGltcG9ydGxpYi5yZWxvYWQoUFApCiAgICBpZiBfbWQ1KFNWMyArICJjb2RlL3A4c19wcm9tcHQucHkiKSAhPSBQUk9NUFRfTUQ1OgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoInA4c19wcm9tcHQucHkgbWQ1IOu2iOydvOy5mCDigJQg7ZSE66Gs7ZSE7Yq46rCAIOuwlOuAjOuptCDquLDspIDshKAg7IiY7LmY6rCAIOuLrOudvOynhOuLpCIpCgogICAgbSA9IHBkLnJlYWRfY3N2KFA5ICsgImhpZGRlbl92Mi9wOHNfYmVoYXZpb3JfdjIuY3N2IikKICAgIG1hbiA9IHBkLnJlYWRfcGFycXVldChTVjMgKyAiZW52L3Jvd19tYW5pZmVzdC5wYXJxdWV0IikKICAgIGFzc2VydCAobWFuLnF1ZXN0aW9uLnZhbHVlcyA9PSBtLnF1ZXN0aW9uLnZhbHVlcykuYWxsKCksICJyb3dfbWFuaWZlc3Qg7KCV66CsIOu2iOydvOy5mCIKICAgIG1bInJpZCJdID0gbnAuYXJhbmdlKGxlbihtKSkKCiAgICBjYiA9IHBkLnJlYWRfcGFycXVldChTVjMgKyAiY2JfdjMucGFycXVldCIpCiAgICBsYWJlbHMgPSBwZC5yZWFkX3BhcnF1ZXQoU1YzICsgImxhYmVsc192My5wYXJxdWV0IikKCiAgICBrdiA9IG0ucXVlc3Rpb24ubWFwKGRpY3QoemlwKGNiLnF1ZXN0aW9uLCBjYi5rc2lkZV92MykpKS5zdHIucmVwbGFjZSgi4oiSIiwgIi0iKS52YWx1ZXMKICAgIGNsID0gbS5jdHhfbGFiZWwuc3RyLnJlcGxhY2UoIuKIkiIsICItIikudmFsdWVzCiAgICB5SyA9IChrdiA9PSAiSysiKS5hc3R5cGUoaW50KQogICAgeUMgPSAoY2wgPT0gIkMrIikuYXN0eXBlKGludCkKCiAgICByZXYgPSBqc29uLmxvYWQob3BlbihTVjMgKyAiZW52L21hbmlmZXN0X2NlbGwxLmpzb24iKSlbIm1vZGVsX3JldmlzaW9uIl0KCiAgICByZXR1cm4gQmVuY2gobT1tLCBtYW49bWFuLCBjYj1jYiwgbGFiZWxzPWxhYmVscywgUFA9UFAsCiAgICAgICAgICAgICAgICAgZm9sZD1tYW4uZm9sZC52YWx1ZXMuYXN0eXBlKGludCksIHlDPXlDLCB5Sz15SywKICAgICAgICAgICAgICAgICBjZWxsPW5wLmFycmF5KFsoIksrIiBpZiBrIGVsc2UgIkstIikgKyAoIkMrIiBpZiBjIGVsc2UgIkMtIikKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgaywgYyBpbiB6aXAoeUssIHlDKV0pLAogICAgICAgICAgICAgICAgIG1vZGVsX3JldmlzaW9uPXJldikKCgpkZWYgbG9hZF9hcm0obmFtZTogc3RyKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJBMCDqsIAg7IOd7ISx7ZWcIO2WieuPmSDqsrDqs7wuIHJpZCwgcmF3LCBwcmVkLCBlbSwgYWNjX3gsIHJldF94LCBoaXRfbGltaXQsIHJlcDMsIGZvbGQuIiIiCiAgICBmID0gQTAgKyBmImdlbl97bmFtZX0ucGFycXVldCIKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhmKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYi7JeG7J2MOiB7Zn1cbkEwIOuFuO2KuOu2gSBBMC00IC8gQTAtNSAvIEEwLTViIOulvCDrqLzsoIAg7Iuk7ZaJ7ZWc64ukLiIpCiAgICBkID0gcGQucmVhZF9wYXJxdWV0KGYpCiAgICBhc3NlcnQgKGQucmlkLnZhbHVlcyA9PSBucC5hcmFuZ2UobGVuKGQpKSkuYWxsKCksIGYie25hbWV9OiByaWQg7KCV66CsIOu2iOydvOy5mCIKICAgIHJldHVybiBkCgoKZGVmIGxvYWRfYXJtX21hdHJpeChhcm1zOiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSk6CiAgICAiIiIoRSwgcHJlZHMsIG5hbWVzKS4gRT0obl9yb3dzLCBuX2FybXMpIDAvMSwgcHJlZHM9KG5fcm93cywgbl9hcm1zKSDrrLjsnpDsl7QuIiIiCiAgICBhcm1zID0gYXJtcyBvciBBUk1TCiAgICBkcyA9IFtsb2FkX2FybShhKSBmb3IgYSBpbiBhcm1zXQogICAgRSA9IG5wLnN0YWNrKFtkLmVtLnZhbHVlcy5hc3R5cGUoZmxvYXQpIGZvciBkIGluIGRzXSwgMSkKICAgIFAgPSBucC5zdGFjayhbZC5wcmVkLnZhbHVlcy5hc3R5cGUob2JqZWN0KSBmb3IgZCBpbiBkc10sIDEpCiAgICByZWYgPSBFWzosIGFybXMuaW5kZXgoUkVGX0FSTSldLm1lYW4oKQogICAgaWYgYWJzKHJlZiAtIDAuNTk5NykgPiAxZS0zOgogICAgICAgIHByaW50KGYiW+qyveqzoF0ge1JFRl9BUk19IEVNIHtyZWY6LjRmfSDigJQgQTAg67O06rOg6rCSIDAuNTk5NyDqs7wg64uk66W064ukIikKICAgIHJldHVybiBFLCBQLCBhcm1zCg=="
FILES["common/scoring.py"] = "IiIi7LGE7KCQLiBBMCDrhbjtirjrtoEgQ2VsbCA2LzEwIOydmCB0cnVuY192MyAvIHNjb3JlX3YzIOyZgCDrj5nsnbztlZjri6QuCgrsoJXqt5ztmZTripQg67CY65Oc7IucIFBQLm5vcm0ocDhzX3Byb21wdCkg7J2EIOyTtOuLpC4gU1F1QUQg7ZGc7KSAIOygleq3nO2ZlOuhnCDrsJTqvrjrqbQKQTAvQTEg67O06rOg6rCS6rO8IOyWtOq4i+uCmCDruYTqtZDqsIAg6rmo7KeE64ukLiDsnbQg7YyM7J287J2AIOqzoOy5mOyngCDslYrripTri6QuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgcmUKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCl9BQkJSID0gciIoPzwhXGJEcikoPzwhXGJNcikoPzwhXGJNcykoPzwhXGJNcnMpKD88IVxiU3QpKD88IVxiSnIpKD88IVxiU3IpKD88IVxiW0EtWl0pIgoKCmRlZiB0cnVuY192MyhyYXcpIC0+IHN0cjoKICAgIHMgPSBzdHIocmF3KS5zdHJpcCgpLnNwbGl0KCJcbiIpWzBdLnN0cmlwKCkKICAgIHMgPSByZS5zdWIociJeKGFuc3dlcnx0aGUgYW5zd2VyIGlzKVxzKjo/XHMqIiwgIiIsIHMsIGZsYWdzPXJlLkkpCiAgICBpZiBsZW4ocy5zcGxpdCgpKSA+IDg6CiAgICAgICAgcyA9IHJlLnNwbGl0KF9BQkJSICsgciJcLlxzKyg/PVtBLVpdKSIsIHMpWzBdCiAgICByZXR1cm4gcmUuc3ViKHIiWy4sOzpcc10rJCIsICIiLCBzLnN0cmlwKCkuc3RyaXAoIlwiJyIpKS5zdHJpcCgpCgoKZGVmIHJlcDModCkgLT4gaW50OgogICAgdyA9IHN0cih0KS5zcGxpdCgpCiAgICBnID0gQ291bnRlcih0dXBsZSh3W2k6aSArIDNdKSBmb3IgaSBpbiByYW5nZShsZW4odykgLSAyKSkKICAgIHJldHVybiBpbnQoYm9vbChnKSBhbmQgbWF4KGcudmFsdWVzKCkpID49IDMpCgoKZGVmIG1ha2Vfc2NvcmVyKGJlbmNoKToKICAgICIiInNjb3JlKGRmKSAtPiBkZi4gZGYg64qUIHJpZCwgcmF3LCBuX3RvayDrpbwg6rCA7KeE64ukLiIiIgogICAgUFAgPSBiZW5jaC5QUAogICAgbSwgY2IgPSBiZW5jaC5tLCBiZW5jaC5jYgogICAgZ2xfcm93ID0gW2pzb24ubG9hZHMoZykgaWYgaXNpbnN0YW5jZShnLCBzdHIpIGVsc2UgbGlzdChnKSBmb3IgZyBpbiBtLmdvbGRdCiAgICBjYnEgPSBkaWN0KHppcChjYi5xdWVzdGlvbiwgY2IuY2JfZ3JlZWR5X3YzKSkKICAgIHN1cmYgPSBtLmFuc3dlcl9zdXJmYWNlLm1hcChQUC5ub3JtKS52YWx1ZXMKICAgIGNibiA9IG0ucXVlc3Rpb24ubWFwKGNicSkubWFwKFBQLm5vcm0pLnZhbHVlcwoKICAgIGRlZiBzY29yZShkOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICBwID0gW1BQLm5vcm0odHJ1bmNfdjMoeCkpIGZvciB4IGluIGQucmF3XQogICAgICAgIGRbInByZWQiXSA9IFt0cnVuY192Myh4KSBmb3IgeCBpbiBkLnJhd10KICAgICAgICBkWyJlbSJdID0gW2ludChhbnkocGkgPT0gUFAubm9ybShnKSBmb3IgZyBpbiBnZykpCiAgICAgICAgICAgICAgICAgICBmb3IgcGksIGdnIGluIHppcChwLCBnbF9yb3cpXQogICAgICAgIGRbImFjY194Il0gPSBbaW50KHMgIT0gIiIgYW5kIHBpID09IHMpIGZvciBwaSwgcyBpbiB6aXAocCwgc3VyZildCiAgICAgICAgZFsicmV0X3giXSA9IFtpbnQoYyAhPSAiIiBhbmQgYyAhPSBzIGFuZCBwaSA9PSBjKQogICAgICAgICAgICAgICAgICAgICAgZm9yIHBpLCBzLCBjIGluIHppcChwLCBzdXJmLCBjYm4pXQogICAgICAgIGRbImhpdF9saW1pdCJdID0gKGQubl90b2sgPj0gUFAuTUFYX05FV19UT0tFTlMpLmFzdHlwZShpbnQpCiAgICAgICAgZFsicmVwMyJdID0gZC5yYXcubWFwKHJlcDMpCiAgICAgICAgcmV0dXJuIGQKCiAgICByZXR1cm4gc2NvcmUKCgpkZWYgZW1fZnJvbV9wcmVkcyhiZW5jaCwgcHJlZHM6IGxpc3Rbc3RyXSkgLT4gbnAubmRhcnJheToKICAgICIiIuydtOuvuCDsnpjrprAgcHJlZCDrrLjsnpDsl7Qg66qp66GdIOKGkiAwLzEgRU0g67Kh7YSwLiIiIgogICAgUFAgPSBiZW5jaC5QUAogICAgZ2xfcm93ID0gW2pzb24ubG9hZHMoZykgaWYgaXNpbnN0YW5jZShnLCBzdHIpIGVsc2UgbGlzdChnKSBmb3IgZyBpbiBiZW5jaC5tLmdvbGRdCiAgICByZXR1cm4gbnAuYXJyYXkoW2ludChhbnkoUFAubm9ybShwKSA9PSBQUC5ub3JtKGcpIGZvciBnIGluIGdnKSkKICAgICAgICAgICAgICAgICAgICAgZm9yIHAsIGdnIGluIHppcChwcmVkcywgZ2xfcm93KV0sIGR0eXBlPWZsb2F0KQoKCmRlZiBzdGFiaWxpdHkoZDogcGQuRGF0YUZyYW1lKSAtPiBkaWN0OgogICAgIiIiQTAg7J2YIGFybSDslYjsoJXshLEg6riw7KSALiDrkZgg64ukIDAuMDUg7J207ZWY7Jes7JW8IOyLoOuisO2VnOuLpC4iIiIKICAgIHJldHVybiB7ImhpdF9saW1pdCI6IGZsb2F0KGQuaGl0X2xpbWl0Lm1lYW4oKSksICJyZXAzIjogZmxvYXQoZC5yZXAzLm1lYW4oKSksCiAgICAgICAgICAgICJzdGFibGUiOiBib29sKGQuaGl0X2xpbWl0Lm1lYW4oKSA8PSAuMDUgYW5kIGQucmVwMy5tZWFuKCkgPD0gLjA1KX0K"
FILES["common/stats.py"] = "IiIi7KeI66y4IOuLqOychCBib290c3RyYXAuIEEwIENlbGwgNyAvIEExIENlbGwgNyDqs7wg64+Z7J287ZWcIOuwqeyLneydtOuLpC4KCuyniOusuOydhCDrs7Xsm5DstpTstpztlZwg65KkIOq3uCDsp4jrrLjsl5Ag7IaN7ZWcIO2WieydhCDsoITrtoAg66qo7JWEIO2Pieq3oOydhCDrgrjri6QKKD0g6rCZ7J2AIOyniOusuOydmCBDKy9DLSDqsIAg7ZWo6ruYIOu9ke2eiOqxsOuCmCDtlajqu5gg67mg7KeE64ukKS4K7ZaJIOuLqOychCDsnqztkZzsp5HsnYAgQysvQy0g7IOB6rSA7J2EIOustOyLnO2VtCBDSSDrpbwg6rO87IaM7LaU7KCV7ZWc64ukLgoK7IKs7KCE65Ox66GdIO2MkOyglQogIOqwnOyEoCAgICAgIDogKOuwqeuylSAtIOq4sOykgCkgzpRFTSA5NSUgQ0kg7ZWY7ZWcID4gMAogIEMrIOu5hOyXtOuTsSA6ICjrsKnrspUgLSBiYXNlKSAgQysgzpRFTSA5NSUgQ0kg7ZWY7ZWcID4gLTIlcCAgIChTQUZFVFkpCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgYmlub210ZXN0CgpTQUZFVFkgPSAtMC4wMgpOX0JPT1QgPSAyMDAwCgoKZGVmIG1ha2VfYm9vdChiZW5jaCwgbl9ib290OiBpbnQgPSBOX0JPT1QsIHNlZWQ6IGludCA9IDApOgogICAgcSA9IGJlbmNoLm0ucXVlc3Rpb24udmFsdWVzCiAgICB1cSA9IG5wLnVuaXF1ZShxKQogICAgcWlkeCA9IHt1OiBucC53aGVyZShxID09IHUpWzBdIGZvciB1IGluIHVxfQogICAgcnMgPSBucC5yYW5kb20uUmFuZG9tU3RhdGUoc2VlZCkKICAgIHJldHVybiBbbnAuY29uY2F0ZW5hdGUoW3FpZHhbdV0gZm9yIHUgaW4gcnMuY2hvaWNlKHVxLCBsZW4odXEpKV0pCiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKG5fYm9vdCldCgoKZGVmIGNpKHY6IG5wLm5kYXJyYXksIGJvb3QsIG1hc2s6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZSk6CiAgICBtYXNrID0gbnAub25lcyhsZW4odiksIGJvb2wpIGlmIG1hc2sgaXMgTm9uZSBlbHNlIG1hc2sKICAgIHMgPSBbdltpXVttYXNrW2ldXS5tZWFuKCkgZm9yIGkgaW4gYm9vdF0KICAgIGxvLCBoaSA9IG5wLnBlcmNlbnRpbGUocywgWzIuNSwgOTcuNV0pCiAgICByZXR1cm4gZmxvYXQobG8pLCBmbG9hdChoaSkKCgpkZWYgZXZhbHVhdGUoYmVuY2gsIGJvb3QsIGVtOiBucC5uZGFycmF5LCBiYXNlOiBucC5uZGFycmF5LAogICAgICAgICAgICAgcmVmOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsIG5hbWU6IHN0ciA9ICIiKSAtPiBkaWN0OgogICAgIiIiZW0vYmFzZS9yZWYg64qUIOuqqOuRkCAobl9yb3dzLCkgMC8xIOuyoe2EsC4iIiIKICAgIGlzQ20gPSBiZW5jaC5pc19jbmVnCiAgICBkID0gZW0gLSBiYXNlCiAgICBkciA9IGQgaWYgcmVmIGlzIE5vbmUgZWxzZSBlbSAtIHJlZgoKICAgIGxvX3AsIGhpX3AgPSBjaShkLCBib290LCB+aXNDbSkKICAgIGxvX3IsIGhpX3IgPSBjaShkciwgYm9vdCkKCiAgICBiMDEgPSBpbnQoKCgocmVmIGlmIHJlZiBpcyBub3QgTm9uZSBlbHNlIGJhc2UpID09IDApICYgKGVtID09IDEpKS5zdW0oKSkKICAgIGIxMCA9IGludCgoKChyZWYgaWYgcmVmIGlzIG5vdCBOb25lIGVsc2UgYmFzZSkgPT0gMSkgJiAoZW0gPT0gMCkpLnN1bSgpKQoKICAgIG91dCA9IGRpY3QoCiAgICAgICAgcG9saWN5PW5hbWUsCiAgICAgICAgRU09ZmxvYXQoZW0ubWVhbigpKSwKICAgICAgICBkRU09ZmxvYXQoZC5tZWFuKCkpLAogICAgICAgIGRDX21pbnVzPWZsb2F0KGRbaXNDbV0ubWVhbigpKSwKICAgICAgICBkQ19wbHVzPWZsb2F0KGRbfmlzQ21dLm1lYW4oKSksCiAgICAgICAgZENfcGx1c19sbz1sb19wLAogICAgICAgIHNhZmU9Ym9vbChsb19wID4gU0FGRVRZKSwKICAgICAgICB2c19yZWY9ZmxvYXQoZHIubWVhbigpKSwgdnNfcmVmX2xvPWxvX3IsIHZzX3JlZl9oaT1oaV9yLAogICAgICAgIGltcHJvdmVkPWJvb2wobG9fciA+IDApLAogICAgICAgIHJlc3RvcmVkPWludCgoKGJhc2UgPT0gMCkgJiAoZW0gPT0gMSkpLnN1bSgpKSwKICAgICAgICBicm9rZW49aW50KCgoYmFzZSA9PSAxKSAmIChlbSA9PSAwKSkuc3VtKCkpLAogICAgICAgIG1jbmVtYXJfcD1mbG9hdChiaW5vbXRlc3QoYjAxLCBiMDEgKyBiMTApLnB2YWx1ZSkgaWYgYjAxICsgYjEwIGVsc2UgMS4wLAogICAgKQogICAgZm9yIGMgaW4gWyJLK0MrIiwgIksrQy0iLCAiSy1DKyIsICJLLUMtIl06CiAgICAgICAgbXNrID0gYmVuY2guY2VsbCA9PSBjCiAgICAgICAgb3V0W2YiZF97Y30iXSA9IGZsb2F0KGRbbXNrXS5tZWFuKCkpIGlmIG1zay5hbnkoKSBlbHNlIGZsb2F0KCJuYW4iKQogICAgb3V0WyJwYXNzIl0gPSBvdXRbImltcHJvdmVkIl0gYW5kIG91dFsic2FmZSJdCiAgICByZXR1cm4gb3V0Cg=="
FILES["common/generate.py"] = "IiIi7IOd7ISxIO2MjOydtO2UhOudvOyduC4gQTAg64W47Yq467aBIEEwLTMg6rO8IOuPmeydvO2VmOqyjCDrp57stpjri6QuCgpTT1RBIOulvCDsg4jroZwg7IOd7ISx7ZWgIOuVjOunjCDsk7Tri6QuIOq4sOykgOyEoCDikaDikaHripQg7IKw7Lac66y8IOyhsO2ajOudvCDsnbQg7YyM7J287J20IO2VhOyalCDsl4bri6QuCgpBMCDsmYAg67CY65Oc7IucIOqwmeyVhOyVvCDtlZjripQg6rKDICjtlZjrgpjrnbzrj4Qg64uk66W066m0IOq4sOykgOyEoOqzvCDruYTqtZAg67aI6rCAKQogIC0gdHJhbnNmb3JtZXJzPT01LjE3LjAsIG1vZGVsIHJldmlzaW9uIOqzoOyglSwgZGV0ZXJtaW5pc3RpYywgdGYzMiBvZmYKICAtIFBQLmJ1aWxkX21lc3NhZ2VzICsgYXBwbHlfY2hhdF90ZW1wbGF0ZSwg7ZaJ67OEIHByb21wdF9tZDUg6rKA7KadCiAgLSBwYWRkaW5nX3NpZGU9J2xlZnQnLCBCUz0xNiwgbl9wcm9tcHRfdG9rIOygleugrAogIC0gRU9TIOyXkCA8fGVvdF9pZHw+IO2PrO2VqCwgbWF4X25ld190b2tlbnMgPSBQUC5NQVhfTkVXX1RPS0VOUwogIC0gc3RlZXJpbmc6IHByZWZpbGwg7J2AIGF0dGVudGlvbiBtYXNrIOuhnCDruYTtjKjrlKkg7Yag7YGwIOyghOyytChQOSksIGRlY29kZSDripQg66eI7KeA66eJIOychOy5mAoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IG9zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKCkJTID0gMTYKCgpkZWYgc2V0dXBfZGV0ZXJtaW5pc20oKSAtPiBOb25lOgogICAgIiIiQ1VEQSDstIjquLDtmZQg7KCE7JeQIO2YuOy2nO2VnOuLpC4gQTAtMSDqs7wg64+Z7J28LiIiIgogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJDVUJMQVNfV09SS1NQQUNFX0NPTkZJRyIsICI6NDA5Njo4IikKICAgIHRvcmNoLnVzZV9kZXRlcm1pbmlzdGljX2FsZ29yaXRobXMoVHJ1ZSwgd2Fybl9vbmx5PVRydWUpCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd190ZjMyID0gRmFsc2UKICAgIHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmFsbG93X3RmMzIgPSBGYWxzZQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gRmFsc2UKCgpjbGFzcyBSdW5uZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYmVuY2gsIGNoZWNrX3ZlcnNpb246IGJvb2wgPSBUcnVlKToKICAgICAgICBpbXBvcnQgdHJhbnNmb3JtZXJzCiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbEZvckNhdXNhbExNLCBBdXRvVG9rZW5pemVyCgogICAgICAgIGlmIGNoZWNrX3ZlcnNpb24gYW5kIHRyYW5zZm9ybWVycy5fX3ZlcnNpb25fXyAhPSAiNS4xNy4wIjoKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgKICAgICAgICAgICAgICAgIGYidHJhbnNmb3JtZXJzIHt0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX199IOKAlCBBMCDripQgNS4xNy4wIOyXkOyEnCDsuKHsoJXtlojri6QuICIKICAgICAgICAgICAgICAgICLrsoTsoITsnbQg64uk66W066m0IOq4sOykgOyEoOqzvCDruYTqtZDtlaAg7IiYIOyXhuuLpC4gIgogICAgICAgICAgICAgICAgIuydmOuPhOyggeycvOuhnCDrhJjquLDroKTrqbQgY2hlY2tfdmVyc2lvbj1GYWxzZS4iKQoKICAgICAgICBzZWxmLmJlbmNoID0gYmVuY2gKICAgICAgICBQUCA9IGJlbmNoLlBQCiAgICAgICAgc2VsZi5QUCA9IFBQCgogICAgICAgIHNlbGYudG9rID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoUFAuTU9ERUxfSUQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXZpc2lvbj1iZW5jaC5tb2RlbF9yZXZpc2lvbikKICAgICAgICBzZWxmLnRvay5wYWRkaW5nX3NpZGUgPSAibGVmdCIKICAgICAgICBpZiBzZWxmLnRvay5wYWRfdG9rZW4gaXMgTm9uZToKICAgICAgICAgICAgc2VsZi50b2sucGFkX3Rva2VuID0gc2VsZi50b2suZW9zX3Rva2VuCgogICAgICAgIHNlbGYubW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgICAgIFBQLk1PREVMX0lELCByZXZpc2lvbj1iZW5jaC5tb2RlbF9yZXZpc2lvbiwgZHR5cGU9dG9yY2guYmZsb2F0MTYsCiAgICAgICAgICAgIGRldmljZV9tYXA9ImF1dG8iLCBhdHRuX2ltcGxlbWVudGF0aW9uPSJzZHBhIikuZXZhbCgpCgogICAgICAgIHNlbGYuZW9zX2lkcyA9IHNvcnRlZCh7c2VsZi50b2suZW9zX3Rva2VuX2lkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi50b2suY29udmVydF90b2tlbnNfdG9faWRzKCI8fGVvdF9pZHw+Iil9IC0ge05vbmV9KQogICAgICAgIHNlbGYuZGVjb2RlX2t3ID0gZGljdChza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNsZWFuX3VwX3Rva2VuaXphdGlvbl9zcGFjZXM9RmFsc2UpCgogICAgICAgIG0sIG1hbiA9IGJlbmNoLm0sIGJlbmNoLm1hbgogICAgICAgIHByb21wdHMgPSBbc2VsZi50b2suYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICAgICAgUFAuYnVpbGRfbWVzc2FnZXMoci5xdWVzdGlvbiwgci5jdHhfdGV4dCksCiAgICAgICAgICAgIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSkgZm9yIHIgaW4gbS5pdGVydHVwbGVzKCldCiAgICAgICAgYmFkID0gc3VtKGhhc2hsaWIubWQ1KHAuZW5jb2RlKCkpLmhleGRpZ2VzdCgpICE9IGgKICAgICAgICAgICAgICAgICAgZm9yIHAsIGggaW4gemlwKHByb21wdHMsIG1hbi5wcm9tcHRfbWQ1KSkKICAgICAgICBpZiBiYWQ6CiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiLtlITroaztlITtirgg7ZW07IucIOu2iOydvOy5mCB7YmFkfe2WiSDigJQgQTAg7JmAIOuLpOuluCDtlITroaztlITtirjri6QiKQogICAgICAgIHNlbGYucHJvbXB0cyA9IG5wLmFycmF5KHByb21wdHMsIGR0eXBlPW9iamVjdCkKCiAgICAgICAgc2VsZi5mb2xkX3Jvd3MgPSB7CiAgICAgICAgICAgIGs6IG1bbWFuLmZvbGQudmFsdWVzID09IGtdCiAgICAgICAgICAgICAgICAgLmFzc2lnbihuX3Byb21wdF90b2s9bWFuLm5fcHJvbXB0X3Rvay52YWx1ZXNbbWFuLmZvbGQudmFsdWVzID09IGtdKQogICAgICAgICAgICAgICAgIC5zb3J0X3ZhbHVlcyhbIm5fcHJvbXB0X3RvayIsICJyaWQiXSwga2luZD0ic3RhYmxlIikucmlkLnZhbHVlcwogICAgICAgICAgICBmb3IgayBpbiByYW5nZSg1KX0KCiAgICAgICAgc2VsZi5zdGVlciA9IF9TdGVlcihzZWxmLm1vZGVsKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIOyDneyEsQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBnZW5lcmF0ZV9yb3dzKHNlbGYsIHJpZHM6IG5wLm5kYXJyYXkpIC0+IGRpY3RbaW50LCB0dXBsZVtzdHIsIGludF1dOgogICAgICAgIG91dCA9IHt9CiAgICAgICAgZm9yIHMgaW4gcmFuZ2UoMCwgbGVuKHJpZHMpLCBCUyk6CiAgICAgICAgICAgIGIgPSByaWRzW3M6cyArIEJTXQogICAgICAgICAgICBlbmMgPSBzZWxmLnRvayhzZWxmLnByb21wdHNbYl0udG9saXN0KCksIHJldHVybl90ZW5zb3JzPSJwdCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIHBhZGRpbmc9VHJ1ZSwgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlKS50byhzZWxmLm1vZGVsLmRldmljZSkKICAgICAgICAgICAgc2VsZi5zdGVlci5tYXNrID0gZW5jLmF0dGVudGlvbl9tYXNrLmNsb25lKCkKICAgICAgICAgICAgZyA9IHNlbGYubW9kZWwuZ2VuZXJhdGUoKiplbmMsIG1heF9uZXdfdG9rZW5zPXNlbGYuUFAuTUFYX05FV19UT0tFTlMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRvX3NhbXBsZT1GYWxzZSwgcGFkX3Rva2VuX2lkPXNlbGYudG9rLnBhZF90b2tlbl9pZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZW9zX3Rva2VuX2lkPXNlbGYuZW9zX2lkcylbOiwgZW5jLmlucHV0X2lkcy5zaGFwZVsxXTpdCiAgICAgICAgICAgIGZvciByLCByb3cgaW4gemlwKGIsIGcudG9saXN0KCkpOgogICAgICAgICAgICAgICAgc3RvcCA9IG5leHQoKGkgZm9yIGksIHQgaW4gZW51bWVyYXRlKHJvdykgaWYgdCBpbiBzZWxmLmVvc19pZHMpLCBOb25lKQogICAgICAgICAgICAgICAgbiA9IHN0b3AgaWYgc3RvcCBpcyBub3QgTm9uZSBlbHNlIGxlbihyb3cpCiAgICAgICAgICAgICAgICBvdXRbaW50KHIpXSA9IChzZWxmLnRvay5kZWNvZGUocm93WzpuXSwgKipzZWxmLmRlY29kZV9rdyksIG4pCiAgICAgICAgcmV0dXJuIG91dAoKICAgIGRlZiBydW5fYWxsKHNlbGYpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICAiIiJmb2xkIOyInOyEnOuMgOuhnCDsoIQg7ZaJIOyDneyEsSDihpIgcmlkIOygleugrCBEYXRhRnJhbWUocmF3LCBuX3RvaywgZm9sZCkuIiIiCiAgICAgICAgcm93cyA9IFtdCiAgICAgICAgZm9yIGsgaW4gcmFuZ2UoNSk6CiAgICAgICAgICAgIG8gPSBzZWxmLmdlbmVyYXRlX3Jvd3Moc2VsZi5mb2xkX3Jvd3Nba10pCiAgICAgICAgICAgIHJvd3MgKz0gW2RpY3QocmlkPXIsIHJhdz14LCBuX3Rvaz1uLCBmb2xkPWspIGZvciByLCAoeCwgbikgaW4gby5pdGVtcygpXQogICAgICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykuc29ydF92YWx1ZXMoInJpZCIpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpjbGFzcyBfU3RlZXI6CiAgICAiIiJBMCDsnZggU3RlZXJBIOyZgCDqsJnsnYAg7Jew7IKwLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbCwgbGF5ZXJzPXJhbmdlKDgsIDE5KSk6CiAgICAgICAgc2VsZi5tb2RlbCA9IG1vZGVsCiAgICAgICAgc2VsZi5sYXllcnMgPSBsaXN0KGxheWVycykKICAgICAgICBzZWxmLnZlYywgc2VsZi5zY2FsZSwgc2VsZi5tYXNrLCBzZWxmLmNhbGxzID0gTm9uZSwge30sIE5vbmUsIHt9CiAgICAgICAgc2VsZi5oID0gW21vZGVsLm1vZGVsLmxheWVyc1tMXS5yZWdpc3Rlcl9mb3J3YXJkX2hvb2soc2VsZi5fbWsoTCkpCiAgICAgICAgICAgICAgICAgIGZvciBMIGluIHNlbGYubGF5ZXJzXQoKICAgIGRlZiBfbWsoc2VsZiwgTCk6CiAgICAgICAgZGVmIGhvb2sobW9kLCBhcmdzLCBvdXQpOgogICAgICAgICAgICBpZiBzZWxmLnZlYyBpcyBOb25lIG9yIEwgbm90IGluIHNlbGYudmVjOgogICAgICAgICAgICAgICAgcmV0dXJuIG91dAogICAgICAgICAgICBoID0gb3V0WzBdIGlmIGlzaW5zdGFuY2Uob3V0LCAodHVwbGUsIGxpc3QpKSBlbHNlIG91dAogICAgICAgICAgICBkID0gKHNlbGYuc2NhbGVbTF0gKiBzZWxmLnZlY1tMXSkudG8oaC5kdHlwZSkKICAgICAgICAgICAgc2VsZi5jYWxsc1tMXSA9IHNlbGYuY2FsbHMuZ2V0KEwsIDApICsgMQogICAgICAgICAgICBpZiBoLnNoYXBlWzFdID4gMToKICAgICAgICAgICAgICAgIGggKz0gc2VsZi5tYXNrWzosIDosIE5vbmVdLnRvKGguZHR5cGUpICogZCAgICAgICMgcHJlZmlsbCAoUDkpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBoWzosIC0xLCA6XSArPSBkICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBkZWNvZGUKICAgICAgICAgICAgcmV0dXJuIG91dAogICAgICAgIHJldHVybiBob29rCgogICAgZGVmIHNldChzZWxmLCB2ZWM9Tm9uZSwgc2NhbGU9Tm9uZSk6CiAgICAgICAgc2VsZi52ZWMgPSBOb25lIGlmIHZlYyBpcyBOb25lIGVsc2UgewogICAgICAgICAgICBMOiB0b3JjaC5hc190ZW5zb3IodiwgZGV2aWNlPXNlbGYubW9kZWwuZGV2aWNlLCBkdHlwZT1zZWxmLm1vZGVsLmR0eXBlKQogICAgICAgICAgICBmb3IgTCwgdiBpbiB2ZWMuaXRlbXMoKX0KICAgICAgICBzZWxmLnNjYWxlLCBzZWxmLmNhbGxzID0gKHNjYWxlIG9yIHt9KSwge30KCiAgICBkZWYgcmVtb3ZlKHNlbGYpOgogICAgICAgIGZvciB4IGluIHNlbGYuaDoKICAgICAgICAgICAgeC5yZW1vdmUoKQo="
FILES["methods/base.py"] = "IiIi66mU7ISc65OcIOyduO2EsO2OmOydtOyKpC4KCuuRkCDsooXrpZjqsIAg7J6I64ukLgogIExvb2t1cE1ldGhvZCA6IEEwIOyCsOy2nOusvOydhCDsobDtmoztlZzri6QuIEdQVSDrtojtlYTsmpQuIOq4sOykgOyEoCDikaDikaHqsIAg7Jes6riwIOyGje2VnOuLpC4KICBHZW5NZXRob2QgICAgOiDsg4jroZwg7IOd7ISx7ZWc64ukLiBHUFUg7ZWE7JqULiBTT1RBIOqwgCDsl6zquLAg7IaN7ZWc64ukLgoK7YyA7JuQ7J20IOuFvOusuOydhCDrp6HsnLzrqbQgR2VuTWV0aG9kIO2VmOuCmOulvCDsg4Hsho3tlbQgbWV0aG9kcy8g7JeQIOy2lOqwgO2VnOuLpC4KY29tbW9uLyDsnYAg6rOg7LmY7KeAIOyViuuKlOuLpC4g6rOg7LmY64qUIOyInOqwhCDshJzroZwg67mE6rWQ6rCAIOu2iOqwgOuKpe2VtOynhOuLpC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAoKUkVHSVNUUlk6IGRpY3Rbc3RyLCB0eXBlXSA9IHt9CgoKZGVmIHJlZ2lzdGVyKG5hbWU6IHN0cik6CiAgICBkZWYgZGVjbyhjbHMpOgogICAgICAgIFJFR0lTVFJZW25hbWVdID0gY2xzCiAgICAgICAgY2xzLm5hbWUgPSBuYW1lCiAgICAgICAgcmV0dXJuIGNscwogICAgcmV0dXJuIGRlY28KCgpjbGFzcyBNZXRob2Q6CiAgICBuYW1lID0gInVubmFtZWQiCiAgICBwYXBlciA9ICIiCiAgICAjOiAnb3VycycgfCAnY29uZmxpY3QtcmVzb2x1dGlvbicgfCAnY29udGV4dC1mYWl0aGZ1bCcKICAgIGZhbWlseSA9ICIiCiAgICBuZWVkc19ncHUgPSBGYWxzZQoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBiZW5jaCk6CiAgICAgICAgc2VsZi5iZW5jaCA9IGJlbmNoCgogICAgZGVmIHJ1bihzZWxmKSAtPiB0dXBsZVtucC5uZGFycmF5LCBsaXN0W3N0cl1dOgogICAgICAgICIiIihlbSDrsqHthLAgKG5fcm93cywpLCBwcmVkIOusuOyekOyXtCDrqqnroZ0pIOuwmO2ZmC4iIiIKICAgICAgICByYWlzZSBOb3RJbXBsZW1lbnRlZEVycm9yCgoKY2xhc3MgTG9va3VwTWV0aG9kKE1ldGhvZCk6CiAgICBuZWVkc19ncHUgPSBGYWxzZQoKCmNsYXNzIEdlbk1ldGhvZChNZXRob2QpOgogICAgIiIi7IOI66GcIOyDneyEse2VmOuKlCDrqZTshJzrk5wuIOuwmOuTnOyLnCBjb21tb24uZ2VuZXJhdGUuUnVubmVyIOuhnCDsg53shLHtlZjqs6AKICAgIGNvbW1vbi5zY29yaW5nLm1ha2Vfc2NvcmVyIOuhnCDssYTsoJDtlZzri6QuIiIiCiAgICBuZWVkc19ncHUgPSBUcnVlCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJlbmNoLCBydW5uZXIpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oYmVuY2gpCiAgICAgICAgc2VsZi5ydW5uZXIgPSBydW5uZXIK"
FILES["methods/ours_lookup.py"] = "IiIi6riw7KSA7ISgIOKRoSBhbHdheXMgc3RlZXJpbmcgKOuwjyBiYXNlLCDquLDtg4AgYXJtKSDigJQgQTAg7IKw7Lac66y8IOyhsO2ajC4KCuyerOq1rO2YhO2VmOyngCDslYrripTri6QuIEEwLTQg7JeQ7IScIO2YhOqyvSBleHAyX1A5X3IwLjA2IOuMgOu5hCDrrLjsnpDsl7Qg67aI7J287LmYIDw9MjAvMTIyNCDroZwK6rKA7Kad7J2EIO2GteqzvO2VnCDsgrDstpzrrLzsnYQg6re464yA66GcIOydveuKlOuLpC4KCiAgYmFzZSAgICAgICAgICAgICAgZ2VuX2Jhc2UucGFycXVldCAgICAgICAgICAgICAgRU0gMC41MzEwCiAgYWx3YXlzX3JldGFpbiAgICAgZ2VuX2xhdGVfcmV0X3IwLjA2LnBhcnF1ZXQgICAgRU0gMC41OTk3ICAoUDksIHJobzAuMDYsIEwxNC0xOCwgd192MykKICBhbHdheXNfcmV0YWluX3IwOSBnZW5fbGF0ZV9yZXRfcjAuMDkucGFycXVldCAgICBFTSAwLjYwMDUgIChDKyDruYTsl7Trk7Eg7Iuk7YyoIOyCrOuhgCkKCkdQVSDrtojtlYTsmpQuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gY29tbW9uLmFydGlmYWN0cyBpbXBvcnQgbG9hZF9hcm0KZnJvbSBjb21tb24uc2NvcmluZyBpbXBvcnQgc3RhYmlsaXR5Cgpmcm9tIC5iYXNlIGltcG9ydCBMb29rdXBNZXRob2QsIHJlZ2lzdGVyCgoKY2xhc3MgX0FybShMb29rdXBNZXRob2QpOgogICAgYXJtID0gImJhc2UiCiAgICBmYW1pbHkgPSAib3VycyIKCiAgICBkZWYgcnVuKHNlbGYpOgogICAgICAgIGQgPSBsb2FkX2FybShzZWxmLmFybSkKICAgICAgICBzdCA9IHN0YWJpbGl0eShkKQogICAgICAgIGlmIG5vdCBzdFsic3RhYmxlIl06CiAgICAgICAgICAgIHByaW50KGYiW+qyveqzoF0ge3NlbGYuYXJtfSDrtojslYjsoJUgIgogICAgICAgICAgICAgICAgICBmIihoaXRfbGltaXQge3N0WydoaXRfbGltaXQnXTouM2Z9LCByZXAzIHtzdFsncmVwMyddOi4zZn0pIikKICAgICAgICByZXR1cm4gZC5lbS52YWx1ZXMuYXN0eXBlKGZsb2F0KSwgbGlzdChkLnByZWQudmFsdWVzKQoKCkByZWdpc3RlcigiYmFzZSIpCmNsYXNzIEJhc2UoX0FybSk6CiAgICBhcm0gPSAiYmFzZSIKICAgIHBhcGVyID0gIuuCtOu2gCAoQTAgZ2VuX2Jhc2UpIgoKCkByZWdpc3RlcigiYWx3YXlzX3JldGFpbiIpCmNsYXNzIEFsd2F5c1JldGFpbihfQXJtKToKICAgIGFybSA9ICJsYXRlX3JldF9yMC4wNiIKICAgIHBhcGVyID0gIuuCtOu2gCAo7ZiE6rK9IFN0ZWVyaW5nX2RldmVsb3AgRXhwMiDrj5nqsrAg7ISk7KCVIC8gQTAg7J6s7ZiEKSIKICAgIGV4cGVjdGVkX2VtID0gMC41OTk3CgoKQHJlZ2lzdGVyKCJhbHdheXNfcmV0YWluX3IwOSIpCmNsYXNzIEFsd2F5c1JldGFpblN0cm9uZyhfQXJtKToKICAgICIiInJobzAuMDkuIEMtIOuKlCArMTYuNjclcCDroZwg642UIOyYpOultOyngOunjCBDKyDtlZjtlZwgLTUuMDclcCDroZwg67mE7Je065OxIOyLpO2MqC4KICAgICfqsJXrj4Trpbwg7Jis66as66Ck66m0IHJvdXRpbmcg7J20IO2VhOyalO2VmOuLpCfripQg7KO87J6l7J2YIOuMgOyhsOq1sC4iIiIKICAgIGFybSA9ICJsYXRlX3JldF9yMC4wOSIKICAgIHBhcGVyID0gIuuCtOu2gCAoQTEgMy0yKSIKICAgIGV4cGVjdGVkX2VtID0gMC42MDA1CgoKQHJlZ2lzdGVyKCJhbHdheXNfZWFybHlfcmV0YWluIikKY2xhc3MgQWx3YXlzRWFybHlSZXRhaW4oX0FybSk6CiAgICBhcm0gPSAiZWFybHlfcmV0X3IwLjA2IgogICAgcGFwZXIgPSAi64K067aAIChBMCkiCgoKZGVmIGNoZWNrX2V4cGVjdGVkKG5hbWU6IHN0ciwgZW06IG5wLm5kYXJyYXksIHRvbDogZmxvYXQgPSA1ZS0zKSAtPiBOb25lOgogICAgZnJvbSAuYmFzZSBpbXBvcnQgUkVHSVNUUlkKICAgIGV4cCA9IGdldGF0dHIoUkVHSVNUUlkuZ2V0KG5hbWUpLCAiZXhwZWN0ZWRfZW0iLCBOb25lKQogICAgaWYgZXhwIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuCiAgICBnb3QgPSBmbG9hdChlbS5tZWFuKCkpCiAgICBmbGFnID0gIk9LIiBpZiBhYnMoZ290IC0gZXhwKSA8PSB0b2wgZWxzZSAi67aI7J287LmYIgogICAgcHJpbnQoZiJb7J6s7ZiEIO2ZleyduF0ge25hbWV9OiBFTSB7Z290Oi40Zn0gdnMg67O06rOg6rCSIHtleHA6LjRmfSAg4oaSIHtmbGFnfSIpCg=="
FILES["methods/ours_router.py"] = "IiIi6riw7KSA7ISgIOKRoCByb3V0aW5nIOKAlCBBMSBsYXRlLW9ubHkgKOydtOuTnSDsmIjsuKEpIHJvdXRlci4KCkExIENlbGwgMn40IOydmCDroZzsp4HsnYQg6re464yA66GcIOyYruqyvOuLpC4gR1BVIOu2iO2VhOyalDoK7ZWcIO2WieyXkOuKlCBlYXJseS9sYXRlIOykkSDtlZjrgpjrp4wg6rCc7J6F7ZWY66+A66GcIGdyZWVkeSDsl5DshJzripQg6rKw6rO87ZGcIOyhsO2ajOqwgCDsi6TsoJwg7IOd7ISx6rO8IOqwmeuLpC4K65Sw65287IScIHJvdXRlciDsnZgg7JiI7LihID0g7ISg7YOd65CcIGFybSDsnZgg7KCA7J6l65CcIHByZWQg66y47J6Q7Je07J2064ukLgoK67O06rOg6rCSOiBFTSAwLjYwNzgsIEFsd2F5cy1yZXRhaW4g64yA67mEICswLjgyJXAgWy0wLjQ5LCArMi4yMV0sIEMtICsxNC41NCwgQysgKzAuODIKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucApmcm9tIHNrbGVhcm4uZGVjb21wb3NpdGlvbiBpbXBvcnQgUENBCmZyb20gc2tsZWFybi5saW5lYXJfbW9kZWwgaW1wb3J0IExvZ2lzdGljUmVncmVzc2lvbiwgUmlkZ2UKZnJvbSBza2xlYXJuLnByZXByb2Nlc3NpbmcgaW1wb3J0IFN0YW5kYXJkU2NhbGVyCgpmcm9tIGNvbW1vbi5hcnRpZmFjdHMgaW1wb3J0IEFSTVMsIFA5LCBSRUZfQVJNLCBsb2FkX2FybV9tYXRyaXgKCmZyb20gLmJhc2UgaW1wb3J0IExvb2t1cE1ldGhvZCwgcmVnaXN0ZXIKClNFRUQsIFBDQV9ESU0gPSAwLCAxMjgKTF9FQVJMWSwgTF9MQVRFID0gNywgMTMgICAgICAgICAgIyDtjJDrj4XsuLUgKDAtYmFzZWQpLiDqsJzsnoXsnYAgTDgtMTIgLyBMMTQtMTgKU0VMX0NQX01JTiA9IC0wLjAxICAgICAgICAgICAgICAgIyDshKDtg50g64uo6rOEIGlubmVyIEMrIM6URU0g7ZWY7ZWcCgpHUklEX01BUkdJTiA9IFstOSwgMC4wLCAwLjAyLCAwLjA1LCAwLjFdCgoKQHJlZ2lzdGVyKCJsYXRlX3JvdXRlciIpCmNsYXNzIExhdGVPbmx5R2FpblJvdXRlcihMb29rdXBNZXRob2QpOgogICAgcGFwZXIgPSAi64K067aAIChBMSBsYXRlLW9ubHkg7J2065OdIHJvdXRlcikiCiAgICBmYW1pbHkgPSAib3VycyIKICAgIGV4cGVjdGVkX2VtID0gMC42MDc4CgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJlbmNoLCBwb2xpY3k6IHN0ciA9ICJsYXRlX2dhaW4iKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGJlbmNoKQogICAgICAgIHNlbGYucG9saWN5ID0gcG9saWN5ICAgICAgICAgICMgJ2xhdGVfZ2FpbicgfCAnZWFybHlfZ2FpbicgfCAnc3RhZ2VkX2dhaW4nCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0g7YyQ64+FIOuqqOuNuAoKICAgIGRlZiBfbG9hZChzZWxmKToKICAgICAgICBiID0gc2VsZi5iZW5jaAogICAgICAgIEUsIFAsIG5hbWVzID0gbG9hZF9hcm1fbWF0cml4KEFSTVMpCiAgICAgICAgc2VsZi5FLCBzZWxmLlAsIHNlbGYubmFtZXMgPSBFLCBQLCBuYW1lcwogICAgICAgIHNlbGYuYmFzZSA9IEVbOiwgbmFtZXMuaW5kZXgoImJhc2UiKV0KICAgICAgICBzZWxmLkcgPSBFIC0gc2VsZi5iYXNlWzosIE5vbmVdCiAgICAgICAgc2VsZi5pRSA9IG5wLmFycmF5KFtpIGZvciBpLCBuIGluIGVudW1lcmF0ZShuYW1lcykgaWYgbi5zdGFydHN3aXRoKCJlYXJseSIpXSkKICAgICAgICBzZWxmLmlMID0gbnAuYXJyYXkoW2kgZm9yIGksIG4gaW4gZW51bWVyYXRlKG5hbWVzKSBpZiBuLnN0YXJ0c3dpdGgoImxhdGUiKV0pCgogICAgICAgIEggPSBucC5sb2FkKFA5ICsgImhpZGRlbl92Mi9wOHNfdjJfaF9vdXQubnB5IiwgbW1hcF9tb2RlPSJyIikKICAgICAgICBzZWxmLlggPSB7IjciOiBucC5hc2FycmF5KEhbOiwgTF9FQVJMWV0pLmFzdHlwZShucC5mbG9hdDMyKSwKICAgICAgICAgICAgICAgICAgIjEzIjogbnAuYXNhcnJheShIWzosIExfTEFURV0pLmFzdHlwZShucC5mbG9hdDMyKX0KICAgICAgICBzZWxmLnEgPSBiLm0ucXVlc3Rpb24udmFsdWVzCgogICAgZGVmIF9maXRfcHJlZGljdChzZWxmLCB0ciwgdGUpIC0+IGRpY3Q6CiAgICAgICAgIiIiQTEgQ2VsbCAzIGZpdF9wcmVkaWN0IOyZgCDrj5nsnbwuIiIiCiAgICAgICAgb3V0ID0ge30KICAgICAgICBmb3IgdGFnIGluICgiNyIsICIxMyIpOgogICAgICAgICAgICBYID0gc2VsZi5YW3RhZ10KICAgICAgICAgICAgc2MgPSBTdGFuZGFyZFNjYWxlcigpLmZpdChYW3RyXSkKICAgICAgICAgICAgcGNhID0gUENBKFBDQV9ESU0sIHJhbmRvbV9zdGF0ZT1TRUVEKS5maXQoc2MudHJhbnNmb3JtKFhbdHJdKSkKICAgICAgICAgICAgWnRyID0gcGNhLnRyYW5zZm9ybShzYy50cmFuc2Zvcm0oWFt0cl0pKQogICAgICAgICAgICBadGUgPSBwY2EudHJhbnNmb3JtKHNjLnRyYW5zZm9ybShYW3RlXSkpCiAgICAgICAgICAgIHpzID0gU3RhbmRhcmRTY2FsZXIoKS5maXQoWnRyKQogICAgICAgICAgICBadHIsIFp0ZSA9IHpzLnRyYW5zZm9ybShadHIpLCB6cy50cmFuc2Zvcm0oWnRlKQoKICAgICAgICAgICAgb3V0WyJwSyIgKyB0YWddID0gTG9naXN0aWNSZWdyZXNzaW9uKEM9MC4xLCBtYXhfaXRlcj0zMDAwKSBcCiAgICAgICAgICAgICAgICAuZml0KFp0ciwgc2VsZi5iZW5jaC55S1t0cl0pLnByZWRpY3RfcHJvYmEoWnRlKVs6LCAxXQogICAgICAgICAgICBpZiB0YWcgPT0gIjEzIjoKICAgICAgICAgICAgICAgIG91dFsicEMxMyJdID0gTG9naXN0aWNSZWdyZXNzaW9uKEM9MC4xLCBtYXhfaXRlcj0zMDAwKSBcCiAgICAgICAgICAgICAgICAgICAgLmZpdChadHIsIHNlbGYuYmVuY2gueUNbdHJdKS5wcmVkaWN0X3Byb2JhKFp0ZSlbOiwgMV0KICAgICAgICAgICAgaWR4ID0gc2VsZi5pRSBpZiB0YWcgPT0gIjciIGVsc2Ugc2VsZi5pTAogICAgICAgICAgICBvdXRbImciICsgdGFnXSA9IFJpZGdlKGFscGhhPTEwMC4wKS5maXQoWnRyLCBzZWxmLkdbdHJdWzosIGlkeF0pLnByZWRpY3QoWnRlKQogICAgICAgIHJldHVybiBvdXQKCiAgICBkZWYgX29vZihzZWxmLCByb3dzLCBuX3NwbGl0OiBpbnQgPSA0LCBzZWVkOiBpbnQgPSBTRUVEKSAtPiBkaWN0OgogICAgICAgIHVxID0gbnAudW5pcXVlKHNlbGYucVtyb3dzXSkKICAgICAgICBybmcgPSBucC5yYW5kb20uUmFuZG9tU3RhdGUoc2VlZCkKICAgICAgICBybmcuc2h1ZmZsZSh1cSkKICAgICAgICBncnAgPSB7dTogaSAlIG5fc3BsaXQgZm9yIGksIHUgaW4gZW51bWVyYXRlKHVxKX0KICAgICAgICBnID0gbnAuYXJyYXkoW2dycFt4XSBmb3IgeCBpbiBzZWxmLnFbcm93c11dKQogICAgICAgIG91dCA9IE5vbmUKICAgICAgICBmb3IgaiBpbiByYW5nZShuX3NwbGl0KToKICAgICAgICAgICAgUGogPSBzZWxmLl9maXRfcHJlZGljdChyb3dzW2cgIT0gal0sIHJvd3NbZyA9PSBqXSkKICAgICAgICAgICAgaWYgb3V0IGlzIE5vbmU6CiAgICAgICAgICAgICAgICBvdXQgPSB7azogbnAuemVyb3MoKGxlbihyb3dzKSwpICsgdi5zaGFwZVsxOl0pIGZvciBrLCB2IGluIFBqLml0ZW1zKCl9CiAgICAgICAgICAgIGZvciBrLCB2IGluIFBqLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBvdXRba11bZyA9PSBqXSA9IHYKICAgICAgICByZXR1cm4gb3V0CgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0g7KCV7LGFCgogICAgZGVmIF9wb2xfZ2FpbihzZWxmLCBQLCBrZXksIGlkeHMsIG1hcmdpbik6CiAgICAgICAgZyA9IFBba2V5XQogICAgICAgIGEgPSBpZHhzW2cuYXJnbWF4KDEpXS5jb3B5KCkKICAgICAgICBhW2cubWF4KDEpIDwgbWFyZ2luXSA9IDAgICAgICAgICAgIyAwID0gYmFzZSA9IG5vLW9wCiAgICAgICAgcmV0dXJuIGEKCiAgICBkZWYgX2FwcGx5KHNlbGYsIFAsIHBybSk6CiAgICAgICAgaWYgc2VsZi5wb2xpY3kgPT0gImVhcmx5X2dhaW4iOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fcG9sX2dhaW4oUCwgImc3Iiwgc2VsZi5pRSwgcHJtWyJtYXJnaW4iXSkKICAgICAgICBpZiBzZWxmLnBvbGljeSA9PSAibGF0ZV9nYWluIjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX3BvbF9nYWluKFAsICJnMTMiLCBzZWxmLmlMLCBwcm1bIm1hcmdpbiJdKQogICAgICAgIGlmIHNlbGYucG9saWN5ID09ICJzdGFnZWRfZ2FpbiI6CiAgICAgICAgICAgIGUgPSBzZWxmLl9wb2xfZ2FpbihQLCAiZzciLCBzZWxmLmlFLCBwcm1bInRlIl0pCiAgICAgICAgICAgIGwgPSBzZWxmLl9wb2xfZ2FpbihQLCAiZzEzIiwgc2VsZi5pTCwgcHJtWyJtYXJnaW4iXSkKICAgICAgICAgICAgcmV0dXJuIG5wLndoZXJlKGUgPiAwLCBlLCBsKQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3Ioc2VsZi5wb2xpY3kpCgogICAgZGVmIF9maXRfcG9saWN5KHNlbGYsIFBfaW4sIHJvd3MpOgogICAgICAgICIiIm5lc3RlZCDshKDtg50uIGlubmVyIEMrIM6URU0gPj0gU0VMX0NQX01JTiDtm4Trs7Qg7Jqw7ISgLCDqt7gg7JWI7JeQ7IScIEVNIOy1nOuMgC4iIiIKICAgICAgICBncmlkID0gKFtkaWN0KG1hcmdpbj1tZykgZm9yIG1nIGluIEdSSURfTUFSR0lOXSBpZiBzZWxmLnBvbGljeSAhPSAic3RhZ2VkX2dhaW4iCiAgICAgICAgICAgICAgICBlbHNlIFtkaWN0KHRlPXRlLCBtYXJnaW49bWcpIGZvciB0ZSBpbiBbMC4wMiwgMC4wNSwgMC4xLCAwLjIsIDldCiAgICAgICAgICAgICAgICAgICAgICBmb3IgbWcgaW4gWy05LCAwLjAsIDAuMDVdXSkKICAgICAgICBiZXN0ID0gTm9uZQogICAgICAgIGlzQ20gPSBzZWxmLmJlbmNoLmlzX2NuZWcKICAgICAgICBmb3IgcHJtIGluIGdyaWQ6CiAgICAgICAgICAgIGEgPSBzZWxmLl9hcHBseShQX2luLCBwcm0pCiAgICAgICAgICAgIGVtID0gc2VsZi5FW3Jvd3MsIGFdLm1lYW4oKQogICAgICAgICAgICBjcCA9IChzZWxmLkVbcm93cywgYV0gLSBzZWxmLmJhc2Vbcm93c10pW35pc0NtW3Jvd3NdXS5tZWFuKCkKICAgICAgICAgICAga2V5ID0gKGNwID49IFNFTF9DUF9NSU4sIGVtKQogICAgICAgICAgICBpZiBiZXN0IGlzIE5vbmUgb3Iga2V5ID4gYmVzdFswXToKICAgICAgICAgICAgICAgIGJlc3QgPSAoa2V5LCBwcm0pCiAgICAgICAgcmV0dXJuIGJlc3RbMV0KCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSDsi6TtlokKCiAgICBkZWYgcnVuKHNlbGYpOgogICAgICAgIHNlbGYuX2xvYWQoKQogICAgICAgIGZvbGQgPSBzZWxmLmJlbmNoLmZvbGQKICAgICAgICBjaG9pY2UgPSBucC56ZXJvcyhzZWxmLmJlbmNoLm4sIGludCkKICAgICAgICBwaWNrZWQgPSBbXQoKICAgICAgICBmb3IgayBpbiByYW5nZSg1KToKICAgICAgICAgICAgdHIgPSBucC53aGVyZShmb2xkICE9IGspWzBdCiAgICAgICAgICAgIHRlID0gbnAud2hlcmUoZm9sZCA9PSBrKVswXQogICAgICAgICAgICBQX2luID0gc2VsZi5fb29mKHRyLCBzZWVkPVNFRUQgKyBrKSAgICAgIyDshKDtg53smqkgKHRlc3Qg66+47IKs7JqpKQogICAgICAgICAgICBQX3RlID0gc2VsZi5fZml0X3ByZWRpY3QodHIsIHRlKSAgICAgICAgIyDsoIHsmqnsmqkKICAgICAgICAgICAgcHJtID0gc2VsZi5fZml0X3BvbGljeShQX2luLCB0cikKICAgICAgICAgICAgY2hvaWNlW3RlXSA9IHNlbGYuX2FwcGx5KFBfdGUsIHBybSkKICAgICAgICAgICAgcGlja2VkLmFwcGVuZCh7ImZvbGQiOiBrLCAqKnBybX0pCgogICAgICAgIHNlbGYuc2VsZWN0ZWQgPSBwaWNrZWQKICAgICAgICBzZWxmLmFjdGlvbl9kaXN0ID0ge3NlbGYubmFtZXNbYV06IGZsb2F0KChjaG9pY2UgPT0gYSkubWVhbigpKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGEgaW4gbnAudW5pcXVlKGNob2ljZSl9CiAgICAgICAgcHJpbnQoIltyb3V0ZXJdIGZvbGTrs4Qg7ISg7YOdOiIsIHBpY2tlZCkKICAgICAgICBwcmludCgiW3JvdXRlcl0g7ZaJ64+ZIOu2hO2PrDoiLAogICAgICAgICAgICAgIHtrOiByb3VuZCh2LCAzKSBmb3IgaywgdiBpbiBzb3J0ZWQoc2VsZi5hY3Rpb25fZGlzdC5pdGVtcygpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAga2V5PWxhbWJkYSB4OiAteFsxXSl9KQoKICAgICAgICByb3dzID0gbnAuYXJhbmdlKHNlbGYuYmVuY2gubikKICAgICAgICBlbSA9IHNlbGYuRVtyb3dzLCBjaG9pY2VdCiAgICAgICAgcHJlZHMgPSBbc2VsZi5QW2ksIGNob2ljZVtpXV0gZm9yIGkgaW4gcm93c10KICAgICAgICByZXR1cm4gZW0uYXN0eXBlKGZsb2F0KSwgcHJlZHMKCgpAcmVnaXN0ZXIoInJlZl9hbHdheXNfZm9yX3JvdXRlciIpCmNsYXNzIF9SZWYoTG9va3VwTWV0aG9kKToKICAgICIiInJvdXRlciDruYTqtZAg6riw7KSAKEFsd2F5cy1yZXRhaW4p7J2EIOqwmeydgCDsobDtmowg6rK966Gc66GcIOq6vOuCtOq4sCDsnITtlZwg67O07KGwLiIiIgogICAgZmFtaWx5ID0gIm91cnMiCgogICAgZGVmIHJ1bihzZWxmKToKICAgICAgICBFLCBQLCBuYW1lcyA9IGxvYWRfYXJtX21hdHJpeChBUk1TKQogICAgICAgIGogPSBuYW1lcy5pbmRleChSRUZfQVJNKQogICAgICAgIHJldHVybiBFWzosIGpdLmFzdHlwZShmbG9hdCksIGxpc3QoUFs6LCBqXSkK"
FILES["methods/ours_regen.py"] = "IiIi6riw7KSA7ISg7J2EIOyCsOy2nOusvCDsobDtmozqsIAg7JWE64uI6528ICoq7KeB7KCRIOyerOyDneyEsSoq7ZWc64ukIChHUFUg7ZWE7JqUKS4KCu2PieyGjOyXkOuKlCDsk7Dsp4Ag7JWK64qU64ukLiBvdXJzX2xvb2t1cC5weSDsnZgg7KGw7ZqMIOqyveuhnOqwgCDsoJXri7XsnbTqs6AsIOq3uOyqveydtCDsnbTrr7gKQTAtNCDsl5DshJwg7ZiE6rK9IOybkOuzuCDrjIDruYQg66y47J6Q7Je0IOu2iOydvOy5mCA8PTIwLzEyMjQg66GcIOqygOymneydhCDthrXqs7ztlojri6QuCgrsnbQg66qo65OI7J20IO2VhOyalO2VnCDqsr3smrDripQg65GY67+Q7J2064ukLgogIDEuIEEwIOyCsOy2nOusvOyXkCDsoJHqt7ztlaAg7IiYIOyXhuydhCDrlYwKICAyLiDtmZjqsr0oR1BVwrfrk5zrnbzsnbTrsoTCt+2MqO2CpOyngCnsnbQg67CU64CM7JeI7J2EIOuVjCDqsJnsnYAg7IiY7LmY6rCAIOuCmOyYpOuKlOyngCDtmZXsnbjtlaAg65WMCgrsnqzsg53shLEg6rKw6rO864qUIEEwIOyCsOy2nOusvOqzvCAqKuusuOyekOyXtCDrtojsnbzsuZggMjDtlokg7J2064K0Kirsl6zslbwg7ZWc64ukLiDrhJjsnLzrqbQg7ZmY6rK97J20IOuLpOultOuLpC4KCuyEpOyglSAoQTAgLyDtmITqsr0gRXhwMiDrj5nqsrApCiAgbGF0ZSAgPSBMMTQtMTguICDrsKntlqXsnYAgd192My5ucHog7J2YIO2CpCBgd19me2t9X0x7TH1gIChrPTAuLjQsIEw9MTQuLjE4KQogIFA5ICAgID0gcHJlZmlsbCDruYTtjKjrlKkg7Yag7YGwIOyghOyytCArIGRlY29kZSDrp4jsp4Drp4kg7JyE7LmYCiAgc2NhbGUgPSByaG8gKiBNRURba11bTF0sICBNRURba11bTF0gPSBtZWRpYW4ofHxIW2ZvbGQgIT0gaywgTF18fCkKICAgICAgICAgIChDLSDtlonrp4zsnbQg7JWE64uI6528IHRyYWluIGZvbGQg7KCE7LK0IO2WiSDquLDspIAuIEEwLTIg7JmAIOuPmeydvCkKICDso7zsnoUgID0gc3Quc2V0KHtMOiBzaWduICogV1trXVtMXX0sIHtMOiByaG8gKiBNRURba11bTF19KQoKZWFybHkoTDgtMTIpIOuwqe2WpeydgCB3X3YzLm5weiDsl5Ag7JeG64ukLiBBMCDripQgcHJvcF9iYWxfZGlmZiDroZwg6re4IOyekOumrOyXkOyEnCDsnqzqs4TsgrDtlZzri6QuCuydtCDrqqjrk4jsnYAgbGF0ZSDrp4wg64uk66Oo66+A66GcIGVhcmx5IGFybSDsnbQg7ZWE7JqU7ZWY66m0IEEwIOuFuO2KuOu2geydhCDsk7Tri6QuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKCmZyb20gY29tbW9uLmFydGlmYWN0cyBpbXBvcnQgU1YzLCBQOSwgbG9hZF9hcm0KZnJvbSBjb21tb24uc2NvcmluZyBpbXBvcnQgbWFrZV9zY29yZXIsIHN0YWJpbGl0eQoKZnJvbSAuYmFzZSBpbXBvcnQgR2VuTWV0aG9kLCByZWdpc3RlcgoKTEFURSA9IGxpc3QocmFuZ2UoMTQsIDE5KSkKRUFSTFkgPSBsaXN0KHJhbmdlKDgsIDEzKSkKCgpkZWYgbG9hZF93X3YzKHZlcmJvc2U6IGJvb2wgPSBUcnVlKSAtPiBkaWN0OgogICAgIiIid192My5ucHog66W8IOydveuKlOuLpC4g7YKkIOq3nOyVveydtCDri6TrpbTrqbQg6re464yA66GcIOuztOyXrCDso7zqs6Ag7KSR64uo7ZWc64ukLiIiIgogICAgeiA9IG5wLmxvYWQoU1YzICsgIndfdjMubnB6IikKICAgIGtleXMgPSBsaXN0KHoua2V5cygpKQogICAgaWYgdmVyYm9zZToKICAgICAgICBwcmludChmIlt3X3YzXSB7bGVuKGtleXMpfeqwnCDtgqQgKOyVniA46rCcKToge2tleXNbOjhdfSIpCiAgICByZXR1cm4ge2s6IHpba10gZm9yIGsgaW4ga2V5c30KCgpkZWYgcGlja19kaXJlY3Rpb24oVzogZGljdCwgZm9sZDogaW50LCBsYXllcjogaW50KSAtPiBucC5uZGFycmF5OgogICAgIiIiZm9sZMK3bGF5ZXIg7JeQIO2VtOuLue2VmOuKlCDrsKntlqUg67Kh7YSwLiDslYzroKTsp4Qg7YKkIOq3nOyVveydhCDssKjroYDroZwg7Iuc64+E7ZWc64ukLiIiIgogICAgayA9IGYid19me2ZvbGR9X0x7bGF5ZXJ9IiAgICAgICAgICAjIEEwLTIg7JeQ7IScIO2ZleyduOuQnCDqt5zslb0KICAgIGlmIGsgaW4gVzoKICAgICAgICByZXR1cm4gV1trXS5hc3R5cGUobnAuZmxvYXQzMikKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICBmIndfdjMubnB6IOyXkCB7a30g6rCAIOyXhuuLpC4g7Iuk7KCcIO2CpCDsmIjsi5w6IHtsaXN0KFcpWzoxMl19XG4iCiAgICAgICAgImxhdGUoTDE0LTE4KSDsmbjsnZgg7Li17J2EIOyalOyyre2WiOqxsOuCmCBucHog6rCAIOuLpOuluCDrsoTsoITsnbTri6QuIikKCgpkZWYgY29tcHV0ZV9tZWQoYmVuY2gsIGZvbGQ6IGludCwgbGF5ZXJzOiBsaXN0W2ludF0pIC0+IGRpY3Q6CiAgICAiIiJ0cmFpbiBmb2xkIO2WieuTpOydmCDsuLXrs4QgfHxofHwg7KSR7JWZ6rCSLiBoaWRkZW5fdjIg7LqQ7Iuc66W8IOyTtOuLpCjsg53shLEg67aI7ZWE7JqUKS4iIiIKICAgIEggPSBucC5sb2FkKFA5ICsgImhpZGRlbl92Mi9wOHNfdjJfaF9vdXQubnB5IiwgbW1hcF9tb2RlPSJyIikKICAgIHRyID0gbnAud2hlcmUoYmVuY2guZm9sZCAhPSBmb2xkKVswXQogICAgb3V0ID0ge30KICAgIGZvciBMIGluIGxheWVyczoKICAgICAgICBoID0gbnAuYXNhcnJheShIW3RyLCBMXSkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgb3V0W0xdID0gZmxvYXQobnAubWVkaWFuKG5wLmxpbmFsZy5ub3JtKGgsIGF4aXM9LTEpKSkKICAgIHJldHVybiBvdXQKCgpjbGFzcyBfUmVnZW4oR2VuTWV0aG9kKToKICAgIGZhbWlseSA9ICJvdXJzIgogICAgbGF5ZXJzID0gTEFURQogICAgcmhvID0gMC4wNgogICAgc2lnbiA9ICsxLjAgICAgICAgICAgIyArMSA9IHJldGFpbiwgLTEgPSBhY2NlcHQKICAgIHJlZl9hcm0gPSAibGF0ZV9yZXRfcjAuMDYiCgogICAgZGVmIHJ1bihzZWxmKToKICAgICAgICBiLCByID0gc2VsZi5iZW5jaCwgc2VsZi5ydW5uZXIKICAgICAgICBXID0gbG9hZF93X3YzKCkKICAgICAgICByb3dzID0gW10KCiAgICAgICAgZm9yIGsgaW4gcmFuZ2UoNSk6CiAgICAgICAgICAgIG1lZCA9IGNvbXB1dGVfbWVkKGIsIGssIHNlbGYubGF5ZXJzKQogICAgICAgICAgICAjIEEwLTIg7J2YIHcg64qUIHByb3BfYmFsX2RpZmYg7JeQ7IScIOydtOuvuCDquLjsnbQgMSDroZwg7KCV6rec7ZmU64+8IOyeiOuLpC4g6re464yA66GcIOyTtOuLpC4KICAgICAgICAgICAgdmVjID0ge0w6IHNlbGYuc2lnbiAqIHBpY2tfZGlyZWN0aW9uKFcsIGssIEwpIGZvciBMIGluIHNlbGYubGF5ZXJzfQogICAgICAgICAgICBzY2FsZSA9IHtMOiBzZWxmLnJobyAqIG1lZFtMXSBmb3IgTCBpbiBzZWxmLmxheWVyc30KCiAgICAgICAgICAgIHIuc3RlZXIuc2V0KHZlYz12ZWMsIHNjYWxlPXNjYWxlKQogICAgICAgICAgICByaWRzID0gci5mb2xkX3Jvd3Nba10KICAgICAgICAgICAgcHJpbnQoZiJbZm9sZCB7a31dIHtsZW4ocmlkcyl97ZaJIHwgTUVEICIKICAgICAgICAgICAgICAgICAgZiJ7IHtMOiByb3VuZChtZWRbTF0sIDEpIGZvciBMIGluIHNlbGYubGF5ZXJzfSB9IikKICAgICAgICAgICAgbyA9IHIuZ2VuZXJhdGVfcm93cyhyaWRzKQogICAgICAgICAgICBhc3NlcnQgYWxsKHIuc3RlZXIuY2FsbHMuZ2V0KEwsIDApID4gMCBmb3IgTCBpbiBzZWxmLmxheWVycyksIFwKICAgICAgICAgICAgICAgICLtm4Ug66+47Zi47LacIOKAlCBzdGVlcmluZyDsnbQg7Iuk7KCc66GcIOyggeyaqeuQmOyngCDslYrslZjri6QiCiAgICAgICAgICAgIHJvd3MgKz0gW2RpY3QocmlkPXJpZCwgcmF3PXgsIG5fdG9rPW4sIGZvbGQ9aykgZm9yIHJpZCwgKHgsIG4pIGluIG8uaXRlbXMoKV0KCiAgICAgICAgci5zdGVlci5zZXQoTm9uZSkKICAgICAgICBkID0gbWFrZV9zY29yZXIoYikocGQuRGF0YUZyYW1lKHJvd3MpLnNvcnRfdmFsdWVzKCJyaWQiKS5yZXNldF9pbmRleChkcm9wPVRydWUpKQoKICAgICAgICBzdCA9IHN0YWJpbGl0eShkKQogICAgICAgIHByaW50KGYiW3tzZWxmLm5hbWV9XSBFTSB7ZC5lbS5tZWFuKCk6LjRmfSB8IGhpdF9saW1pdCB7c3RbJ2hpdF9saW1pdCddOi4zZn0gIgogICAgICAgICAgICAgIGYifCByZXAzIHtzdFsncmVwMyddOi4zZn0iKQogICAgICAgIHNlbGYuY29tcGFyZV90b19hcnRpZmFjdChkKQogICAgICAgIHJldHVybiBkLmVtLnZhbHVlcy5hc3R5cGUoZmxvYXQpLCBsaXN0KGQucHJlZC52YWx1ZXMpCgogICAgZGVmIGNvbXBhcmVfdG9fYXJ0aWZhY3Qoc2VsZiwgZDogcGQuRGF0YUZyYW1lKSAtPiBOb25lOgogICAgICAgICIiIkEwIOyCsOy2nOusvOqzvCDrjIDsobAuIDIw7ZaJIOuEmOqyjCDri6TrpbTrqbQg7ZmY6rK97J20IOuLpOuluCDqsoPsnbTri6QuIiIiCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZWYgPSBsb2FkX2FybShzZWxmLnJlZl9hcm0pCiAgICAgICAgZXhjZXB0IChTeXN0ZW1FeGl0LCBSdW50aW1lRXJyb3IpOgogICAgICAgICAgICBwcmludCgiICAoQTAg7IKw7Lac66y8IOyXhuydjCDigJQg64yA7KGwIOyDneuetSkiKQogICAgICAgICAgICByZXR1cm4KICAgICAgICBtaXNtID0gaW50KChkLnByZWQudmFsdWVzICE9IHJlZi5wcmVkLnZhbHVlcykuc3VtKCkpCiAgICAgICAgZGVtID0gZmxvYXQoZC5lbS5tZWFuKCkgLSByZWYuZW0ubWVhbigpKQogICAgICAgIHZlcmRpY3QgPSAiT0siIGlmIG1pc20gPD0gMjAgZWxzZSAi67aI7J287LmYIOKAlCDtmZjqsr0g7ZmV7J24IO2VhOyalCIKICAgICAgICBwcmludChmIiAgQTAg64yA7KGwOiDrrLjsnpDsl7Qg67aI7J287LmYIHttaXNtfS97bGVuKGQpfSB8IM6URU0ge2RlbTorLjRmfSDihpIge3ZlcmRpY3R9IikKCgpAcmVnaXN0ZXIoInJlZ2VuX2Jhc2UiKQpjbGFzcyBSZWdlbkJhc2UoX1JlZ2VuKToKICAgICIiIuqwnOyehSDsl4bsnYwuIOyDneyEsSDtjIzsnbTtlITrnbzsnbgg7J6Q7LK06rCAIEEwIOyZgCDqsJnsnYDsp4Ag7ZmV7J247ZWY64qUIOq4sOykgC4iIiIKICAgIHBhcGVyID0gIuuCtOu2gCAoQTAgYmFzZSDsnqzsg53shLEpIgogICAgcmhvID0gMC4wCiAgICByZWZfYXJtID0gImJhc2UiCgogICAgZGVmIHJ1bihzZWxmKToKICAgICAgICBiLCByID0gc2VsZi5iZW5jaCwgc2VsZi5ydW5uZXIKICAgICAgICByLnN0ZWVyLnNldChOb25lKQogICAgICAgIGQgPSBtYWtlX3Njb3JlcihiKShyLnJ1bl9hbGwoKSkKICAgICAgICBwcmludChmIltyZWdlbl9iYXNlXSBFTSB7ZC5lbS5tZWFuKCk6LjRmfSIpCiAgICAgICAgc2VsZi5jb21wYXJlX3RvX2FydGlmYWN0KGQpCiAgICAgICAgcmV0dXJuIGQuZW0udmFsdWVzLmFzdHlwZShmbG9hdCksIGxpc3QoZC5wcmVkLnZhbHVlcykKCgpAcmVnaXN0ZXIoInJlZ2VuX2Fsd2F5c19yZXRhaW4iKQpjbGFzcyBSZWdlbkFsd2F5c1JldGFpbihfUmVnZW4pOgogICAgIiIi4pGhIGFsd2F5cyBzdGVlcmluZyDsnqzsg53shLEuIOuztOqzoOqwkiBFTSAwLjU5OTcuIiIiCiAgICBwYXBlciA9ICLrgrTrtoAgKO2YhOqyvSBFeHAyIOuPmeqysCDshKTsoJUg7J6s7IOd7ISxKSIKICAgIGV4cGVjdGVkX2VtID0gMC41OTk3CiAgICBsYXllcnMsIHJobywgc2lnbiwgcmVmX2FybSA9IExBVEUsIDAuMDYsICsxLjAsICJsYXRlX3JldF9yMC4wNiIKCgpAcmVnaXN0ZXIoInJlZ2VuX2Fsd2F5c19yZXRhaW5fcjA5IikKY2xhc3MgUmVnZW5BbHdheXNSZXRhaW5SMDkoX1JlZ2VuKToKICAgICIiInJobzAuMDkuIEMrIOu5hOyXtOuTsSDsi6TtjKgg64yA7KGw6rWwLiDrs7Tqs6DqsJIgRU0gMC42MDA1LiIiIgogICAgcGFwZXIgPSAi64K067aAIChBMSAzLTIg7J6s7IOd7ISxKSIKICAgIGV4cGVjdGVkX2VtID0gMC42MDA1CiAgICBsYXllcnMsIHJobywgc2lnbiwgcmVmX2FybSA9IExBVEUsIDAuMDksICsxLjAsICJsYXRlX3JldF9yMC4wOSIK"
FILES["methods/sota_contrastive.py"] = "IiIi66Gc7KeTIOyCsOyIoCDqs4Tsl7QgU09UQSAoQ0FEIC8gQWRhQ0FEKS4KCuydtCDqs4Tsl7TsnYAgY29udGV4dC1mYWl0aGZ1bCDqsIDsobHsnbTri6QuIOybkOuFvOusuOydgCDrs4DsobDrkJwg66y466el7J2YIOuLteydhCBnb2xkIOuhnCDrs7Tsp4Drp4wsCuuzuCDtlZjrhKTsiqTripQgcDhzIOydmCBnb2xkKOyCrOyLpCnroZwg7LGE7KCQ7ZWc64ukLiDrlLDrnbzshJwgQy0g7JeQ7IScIGJhc2Ug67O064ukIOuCruqyjCDrgpjsmKTripQg6rKD7J20CuygleyDgeydtOupsCwg6re4IOyekOyytOqwgCAn6rO87KCcIOyEpOygleydtCDri6TrpbTri6Qn64qUIOymneqxsOuLpC4g7J206riw6riwIOychO2VnCDruYTqtZDqsIAg7JWE64uI64ukLgoKICBDQUQgICAgKFNoaSBldCBhbC4sIDIwMjMpICA6IGxvZ2l0ID0gKDErYSkqbG9naXRfY3R4IC0gYSpsb2dpdF9jYiwgIGE9MS4wCiAgQWRhQ0FEIChXYW5nIGV0IGFsLiwgMjAyNCkgOiBhIOulvCBKU0QocF9jdHggfHwgcF9jYikg66GcIO2WieuniOuLpCDsoIHsnZEg7KGw7KCICgrrkZgg64ukIENCIO2MqOyKpOulvCDrlLDroZwg64+M66as66+A66GcIDItcGFzcyDri6QuIOu5hOyaqSDruYTqtZAg7LaV7JeQIOq3uOuMgOuhnCDsk7gg7IiYIOyeiOuLpC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgpmcm9tIGNvbW1vbi5zY29yaW5nIGltcG9ydCBtYWtlX3Njb3JlcgoKZnJvbSAuYmFzZSBpbXBvcnQgR2VuTWV0aG9kLCByZWdpc3RlcgoKCmRlZiBfanNkKHA6IHRvcmNoLlRlbnNvciwgcTogdG9yY2guVGVuc29yKSAtPiBmbG9hdDoKICAgIG0gPSAwLjUgKiAocCArIHEpCiAgICBrbCA9IGxhbWJkYSBhLCBiOiAoYSAqIChhLmNsYW1wX21pbigxZS0xMikubG9nKCkgLSBiLmNsYW1wX21pbigxZS0xMikubG9nKCkpKS5zdW0oKQogICAgcmV0dXJuIGZsb2F0KDAuNSAqIGtsKHAsIG0pICsgMC41ICoga2wocSwgbSkpCgoKY2xhc3MgX0NvbnRyYXN0aXZlKEdlbk1ldGhvZCk6CiAgICBhZGFwdGl2ZSA9IEZhbHNlCiAgICBhbHBoYSA9IDEuMAogICAgZmFtaWx5ID0gImNvbnRleHQtZmFpdGhmdWwiCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIF9vbmUoc2VsZiwgcmlkOiBpbnQpIC0+IHR1cGxlW3N0ciwgaW50XToKICAgICAgICByID0gc2VsZi5ydW5uZXIKICAgICAgICB0b2ssIG1vZGVsLCBQUCA9IHIudG9rLCByLm1vZGVsLCByLlBQCiAgICAgICAgcm93ID0gc2VsZi5iZW5jaC5tLmlsb2NbcmlkXQoKICAgICAgICBwX2N0eCA9IHIucHJvbXB0c1tyaWRdCiAgICAgICAgcF9jYiA9IHRvay5hcHBseV9jaGF0X3RlbXBsYXRlKFBQLmJ1aWxkX21lc3NhZ2VzKHJvdy5xdWVzdGlvbiwgTm9uZSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSkKCiAgICAgICAgaWRzX2N0eCA9IHRvayhwX2N0eCwgcmV0dXJuX3RlbnNvcnM9InB0IiwgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlKS50byhtb2RlbC5kZXZpY2UpCiAgICAgICAgaWRzX2NiID0gdG9rKHBfY2IsIHJldHVybl90ZW5zb3JzPSJwdCIsIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSkudG8obW9kZWwuZGV2aWNlKQoKICAgICAgICBjdXJfY3R4LCBjdXJfY2IgPSBpZHNfY3R4LmlucHV0X2lkcywgaWRzX2NiLmlucHV0X2lkcwogICAgICAgIHBhc3RfY3R4ID0gcGFzdF9jYiA9IE5vbmUKICAgICAgICBvdXRfaWRzOiBsaXN0W2ludF0gPSBbXQoKICAgICAgICBmb3IgXyBpbiByYW5nZShQUC5NQVhfTkVXX1RPS0VOUyk6CiAgICAgICAgICAgIG9fY3R4ID0gbW9kZWwoaW5wdXRfaWRzPWN1cl9jdHgsIHBhc3Rfa2V5X3ZhbHVlcz1wYXN0X2N0eCwgdXNlX2NhY2hlPVRydWUpCiAgICAgICAgICAgIG9fY2IgPSBtb2RlbChpbnB1dF9pZHM9Y3VyX2NiLCBwYXN0X2tleV92YWx1ZXM9cGFzdF9jYiwgdXNlX2NhY2hlPVRydWUpCiAgICAgICAgICAgIHBhc3RfY3R4LCBwYXN0X2NiID0gb19jdHgucGFzdF9rZXlfdmFsdWVzLCBvX2NiLnBhc3Rfa2V5X3ZhbHVlcwoKICAgICAgICAgICAgbF9jdHggPSBvX2N0eC5sb2dpdHNbMCwgLTFdLmZsb2F0KCkKICAgICAgICAgICAgbF9jYiA9IG9fY2IubG9naXRzWzAsIC0xXS5mbG9hdCgpCiAgICAgICAgICAgIGEgPSBfanNkKEYuc29mdG1heChsX2N0eCwgLTEpLCBGLnNvZnRtYXgobF9jYiwgLTEpKSBpZiBzZWxmLmFkYXB0aXZlIGVsc2Ugc2VsZi5hbHBoYQoKICAgICAgICAgICAgbnh0ID0gaW50KCgoMSArIGEpICogbF9jdHggLSBhICogbF9jYikuYXJnbWF4KCkpCiAgICAgICAgICAgIGlmIG54dCBpbiByLmVvc19pZHM6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBvdXRfaWRzLmFwcGVuZChueHQpCiAgICAgICAgICAgIHQgPSB0b3JjaC50ZW5zb3IoW1tueHRdXSwgZGV2aWNlPW1vZGVsLmRldmljZSkKICAgICAgICAgICAgY3VyX2N0eCA9IGN1cl9jYiA9IHQKCiAgICAgICAgcmV0dXJuIHRvay5kZWNvZGUob3V0X2lkcywgKipyLmRlY29kZV9rdyksIGxlbihvdXRfaWRzKQoKICAgIGRlZiBydW4oc2VsZik6CiAgICAgICAgcm93cyA9IFtdCiAgICAgICAgZm9yIHJpZCBpbiByYW5nZShzZWxmLmJlbmNoLm4pOgogICAgICAgICAgICByYXcsIG4gPSBzZWxmLl9vbmUocmlkKQogICAgICAgICAgICByb3dzLmFwcGVuZChkaWN0KHJpZD1yaWQsIHJhdz1yYXcsIG5fdG9rPW4sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9sZD1pbnQoc2VsZi5iZW5jaC5mb2xkW3JpZF0pKSkKICAgICAgICAgICAgaWYgKHJpZCArIDEpICUgMTAwID09IDA6CiAgICAgICAgICAgICAgICBwcmludChmIiAge3JpZCsxfS97c2VsZi5iZW5jaC5ufSIpCiAgICAgICAgZCA9IG1ha2Vfc2NvcmVyKHNlbGYuYmVuY2gpKHBkLkRhdGFGcmFtZShyb3dzKSkKICAgICAgICByZXR1cm4gZC5lbS52YWx1ZXMuYXN0eXBlKGZsb2F0KSwgbGlzdChkLnByZWQudmFsdWVzKQoKCkByZWdpc3RlcigiY2FkIikKY2xhc3MgQ0FEKF9Db250cmFzdGl2ZSk6CiAgICBwYXBlciA9ICJUcnVzdGluZyBZb3VyIEV2aWRlbmNlIChTaGkgZXQgYWwuLCBhclhpdiAyMzA1LjE0NzM5KSIKICAgIGFkYXB0aXZlID0gRmFsc2UKCiAgICBkZWYgX19pbml0X18oc2VsZiwgYmVuY2gsIHJ1bm5lciwgYWxwaGE6IGZsb2F0ID0gMS4wKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGJlbmNoLCBydW5uZXIpCiAgICAgICAgc2VsZi5hbHBoYSA9IGFscGhhCgoKQHJlZ2lzdGVyKCJhZGFjYWQiKQpjbGFzcyBBZGFDQUQoX0NvbnRyYXN0aXZlKToKICAgIHBhcGVyID0gIkFkYUNBRCAoV2FuZyBldCBhbC4sIDIwMjQpIgogICAgYWRhcHRpdmUgPSBUcnVlCg=="
FILES["methods/sota_spare.py"] = "IiIiU3BBUkUg4oCUIFNBRS1CYXNlZCBSZXByZXNlbnRhdGlvbiBFbmdpbmVlcmluZyAoWmhhbyBldCBhbC4sIE5BQUNMIDIwMjUgbG9uZyAyNjQpLgpodHRwczovL2FyeGl2Lm9yZy9hYnMvMjQxMC4xNTk5OSAgLyAgZ2l0aHViLmNvbS95dXpoYW91b2UvU0FFLWJhc2VkLXJlcHJlc2VudGF0aW9uLWVuZ2luZWVyaW5nCgrsm5DrhbzrrLgg7KCI7LCoKMKnNCnrpbwg6re464yA66GcIOyYruqyvOuLpC4KCiAgNC4xIOy2qeuPjCDsnbjsiqTthLTsiqQo7Jqw66as7J2YIEPiiJIg7ZaJKeulvCDrqqjrjbgg7Lac66ClIOq4sOykgOycvOuhnCDrkZjroZwg64KY64iI64ukLgogICAgICBEX0MgPSDrrLjrp6Ug64u17J2EIOyDneyEse2VnCDtlokgKOyasOumrCBhY2NfeD09MSkKICAgICAgRF9NID0g7YyM652866mU7Yq466atIOuLteydhCDsg53shLHtlZwg7ZaJICjsmrDrpqwgcmV0X3g9PTEpCiAgICAgIOqwgSDqt7jro7nsnZgg66eI7KeA66eJIOyeheugpSDthqDtgbAgaGlkZGVuIHN0YXRlIOulvCBTQUUg66GcIOyduOy9lOuUqe2VtCDtj4nqt6Ag4oaSIHrMhF9DLCB6zIRfTQogIDQuMiDqsIEgU0FFIO2ZnOyEseqzvCDtlonrj5kg652867KoIOqwhCDsg4HtmLjsoJXrs7Trn4kgSShaX2k7WSkg7J2EIOq1rO2VtCwg64iE7KCBIOu5hOycqOydtCBLIOqwgCDrkKAg65WM6rmM7KeACiAgICAgIOyDgeychCDtmZzshLHsnYQg6rOg66W464ukLiBFX0NbWl9pXSDiiJIgRV9NW1pfaV0g67aA7Zi466GcIOyWtOuKkCDtlonrj5nsl5Ag7IaN7ZWY64qU7KeAIOqwgOuluOuLpC4KICAgICAgICB6X0NpID0gesyEX0NpICAo7LCo7J20ID4gMCksICB6X01pID0gesyEX01pICAo7LCo7J20IDwgMCksICDrgpjrqLjsp4AgMAogIDQuMyDstpTroaAg7IucIGgg7J2YIFNBRSDtmZzshLEgeiDsl5Ag64yA7ZW0CiAgICAgICAgeuKBuyA9IG1pbih6LCB6X3VuZGVzaXJlZCksICB64oG6ID0gbWF4KHpfZGVzaXJlZCDiiJIgeiwgMCkKICAgICAgICBoJyA9IGggKyDOsSjiiJJnKHrigbspICsgZyh64oG6KSkKICAgICAg65SU7L2U642UIGJpYXMg64qUIOyDgeyHhOuQmOuvgOuhnCBnKHopID0geiBAIFdfZGVjIOuhnCDstqnrtoTtlZjri6QuCgrsm5DrhbzrrLgg7ZWY7J207Y287YyM652866+47YSwKExsYW1hMy04Qik6IDEzfjE27Li1LCDOsT0yLjAsIEs9MC4wNy4KCuyasOumrCDshKTsoJXqs7zsnZgg64yA7J2RCiAgc3RlZXI9Im1lbW9yeSIgID0g7YyM652866mU7Yq466atIOyCrOyaqSA9IOyasOumrCByZXRhaW4gIOKGkiBhbHdheXNfcmV0YWluIC8gbGF0ZV9yb3V0ZXIg7JmAIOqwmeydgCDrsKntlqUKICBzdGVlcj0iY29udGV4dCIgPSDrrLjrp6Ug7IKs7JqpICAgICAgID0g7Jqw66asIGFjY2VwdCAg4oaSIOuwmOuMgCDrsKntlqUuIOyWkeyqvSDri6Qg7Iuj64qU64ukLgoK7J2Y64+E7KCB7Jy866GcIOuLpOultOqyjCDtlZwg7KCQKOuztOqzoOyEnOyXkCDrqoXsi5ztlaAg6rKDKQogIDEuIOybkOuFvOusuOydgCAzLXNob3Qg642w66qo66GcIOuLtSDtmJXsi53snYQg66ee7LaY64ukLiDsmrDrpqzripQgQTAg7ZSE66Gs7ZSE7Yq4KHplcm8tc2hvdCnrpbwg7JO064ukLgogICAgIO2UhOuhrO2UhO2KuOulvCDrsJTqvrjrqbQg6riw7KSA7ISg6rO8IOu5hOq1kOqwgCDrtojqsIDriqXtlbTsp4Drr4DroZwg7ZWY64Sk7IqkIOyqveydhCDsmrDshKDtlojri6QuCiAgMi4g7JuQ64W866y47J2YIGNvbmZpZGVuY2Ug6rCA7KSRIO2Pieq3oChBcHBlbmRpeCBDLjEp7J2AIOq4sOuzuOqwkuyXkOyEnCDrgYTqs6Ag64uo7IicIO2Pieq3oOydhCDsk7Tri6QuCiAgICAg64W866y47J20ICJzbGlnaHQgaW1wcm92ZW1lbnQiIOudvCDsoIHsl4jqs6AsIOqzteyLnSDOuz1sb2dQKEMpLyhsb2dQKEMpK2xvZ1AoTSkpIOydgAogICAgIOuRkCDqsJLsnbQg66qo65GQIOydjOyImOudvCDrjZQg7ZmV7Iug7ZWg7IiY66GdIOqwgOykkey5mOqwgCDrgq7slYTsp4DripQg7ZiV7YOc7Jes7IScIOq3uOuMgOuhnCDsmK7quLDquLAg7Ja066C164ukLgogICAgIHdlaWdodGluZz0icGFwZXIiIOuhnCDsvKQg7IiYIOyeiOqyjOuKlCDtlbQg65GQ7JeI64ukLgogIDMuIFNBRSDripQgRWxldXRoZXJBSS9zYWUtbGxhbWEtMy4xLThiLTMyeCAoYmFzZSDrqqjrjbgg7ZWZ7Iq167O4KSDrpbwgSW5zdHJ1Y3Qg7JeQIOyggeyaqe2VnOuLpC4KICAgICDrtojsnbzsuZjsnbTrr4DroZwgRlZVIOuhnCDsnqzqtazshLEg7ZKI7KeI7J2EIOuovOyggCDtmZXsnbjtlZjqs6AsIOuCmOyBmOuptCDspJHri6jtlZzri6QuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSBza2xlYXJuLmZlYXR1cmVfc2VsZWN0aW9uIGltcG9ydCBtdXR1YWxfaW5mb19jbGFzc2lmCgpmcm9tIGNvbW1vbi5hcnRpZmFjdHMgaW1wb3J0IGxvYWRfYXJtCmZyb20gY29tbW9uLnNjb3JpbmcgaW1wb3J0IG1ha2Vfc2NvcmVyCgpmcm9tIC5iYXNlIGltcG9ydCBHZW5NZXRob2QsIHJlZ2lzdGVyCgpTQUVfUkVQTyA9ICJFbGV1dGhlckFJL3NhZS1sbGFtYS0zLjEtOGItMzJ4IgpMQVlFUlMgPSAoMTMsIDE0LCAxNSwgMTYpICAgICAgIyDsm5DrhbzrrLggTGxhbWEzLThCIOyEpOyglQpBTFBIQSA9IDIuMApLX1BST1AgPSAwLjA3CkZWVV9NQVggPSAwLjM1ICAgICAgICAgICAgICAgICAjIOyerOq1rOyEsSDtkojsp4gg7ZWY7ZWcLiDrhJjsnLzrqbQgaG9va3BvaW50IOu2iOydvOy5mOulvCDsnZjsi6ztlZzri6QuCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIFNBRSDslrTrjJHthLAKCmNsYXNzIF9TQUU6CiAgICAiIiJzcGFyc2lmeShFbGV1dGhlckFJKSBTQUUg656Y7Y28LiBlbmNvZGUg4oaSIChpbmRpY2VzLCB2YWx1ZXMpLCBkZWNvZGUg4oaSIFdfZGVjIO2WiS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcmVwbzogc3RyLCBsYXllcjogaW50LCBkZXZpY2UsIGR0eXBlPXRvcmNoLmZsb2F0MzIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBzcGFyc2lmeSBpbXBvcnQgU2FlCiAgICAgICAgZXhjZXB0IEltcG9ydEVycm9yIGFzIGU6CiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoInBpcCBpbnN0YWxsIHNwYXJzaWZ5IOqwgCDtlYTsmpTtlZjri6QiKSBmcm9tIGUKCiAgICAgICAgbGFzdCA9IE5vbmUKICAgICAgICBmb3IgaHAgaW4gKGYibGF5ZXJzLntsYXllcn0iLCBmIm1vZGVsLmxheWVycy57bGF5ZXJ9IiwgZiJoLntsYXllcn0iKToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2VsZi5zYWUgPSBTYWUubG9hZF9mcm9tX2h1YihyZXBvLCBob29rcG9pbnQ9aHApLnRvKGRldmljZSkudG8oZHR5cGUpCiAgICAgICAgICAgICAgICBzZWxmLmhvb2twb2ludCA9IGhwCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAgICAgICAgICAgICAgICAgICAgICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgICAgICBsYXN0ID0gZQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJ7cmVwb30g7JeQ7IScIGxheWVyIHtsYXllcn0gaG9va3BvaW50IOulvCDssL7sp4Ag66q77ZaI64ukOiB7bGFzdH0iKQoKICAgICAgICBzZWxmLmRldmljZSwgc2VsZi5kdHlwZSA9IGRldmljZSwgZHR5cGUKICAgICAgICBzZWxmLm5fbGF0ZW50cyA9IHNlbGYuc2FlLldfZGVjLnNoYXBlWzBdCiAgICAgICAgc2VsZi5XX2RlYyA9IHNlbGYuc2FlLldfZGVjLmRldGFjaCgpICAgICAgICAgICAgIyAobl9sYXRlbnRzLCBkKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBlbmNvZGUoc2VsZiwgaDogdG9yY2guVGVuc29yKToKICAgICAgICAiIiJoOiAoQiwgZCkg4oaSIChpZHggKEIsaykgaW50NjQsIHZhbCAoQixrKSBmbG9hdCkiIiIKICAgICAgICBvdXQgPSBzZWxmLnNhZS5lbmNvZGUoaC50byhzZWxmLmRldmljZSwgc2VsZi5kdHlwZSkpCiAgICAgICAgcmV0dXJuIG91dC50b3BfaW5kaWNlcy5sb25nKCksIG91dC50b3BfYWN0cy5mbG9hdCgpCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIGZ2dShzZWxmLCBoOiB0b3JjaC5UZW5zb3IpIC0+IGZsb2F0OgogICAgICAgICIiImZyYWN0aW9uIG9mIHZhcmlhbmNlIHVuZXhwbGFpbmVkLiBob29rcG9pbnQg6rCAIO2LgOumrOuptCAxLjAg6re87LKY6rCAIOuQnOuLpC4iIiIKICAgICAgICBpZHgsIHZhbCA9IHNlbGYuZW5jb2RlKGgpCiAgICAgICAgcmVjID0gdG9yY2guemVyb3NfbGlrZShoLCBkdHlwZT1zZWxmLmR0eXBlLCBkZXZpY2U9c2VsZi5kZXZpY2UpCiAgICAgICAgZm9yIGogaW4gcmFuZ2UoaC5zaGFwZVswXSk6CiAgICAgICAgICAgIHJlY1tqXSA9IHZhbFtqXS50byhzZWxmLmR0eXBlKSBAIHNlbGYuV19kZWNbaWR4W2pdXQogICAgICAgIHJlYyA9IHJlYyArIHNlbGYuc2FlLmJfZGVjLmRldGFjaCgpCiAgICAgICAgaCA9IGgudG8oc2VsZi5kZXZpY2UsIHNlbGYuZHR5cGUpCiAgICAgICAgcmV0dXJuIGZsb2F0KCgoaCAtIHJlYykgKiogMikuc3VtKCkgLyAoKGggLSBoLm1lYW4oMCkpICoqIDIpLnN1bSgpKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSDrs7jssrQKCmNsYXNzIF9TcEFSRShHZW5NZXRob2QpOgogICAgcGFwZXIgPSAiU3BBUkUgKFpoYW8gZXQgYWwuLCBOQUFDTCAyMDI1KSIKICAgIGZhbWlseSA9ICJjb25mbGljdC1yZXNvbHV0aW9uIgogICAgc3RlZXIgPSAibWVtb3J5IgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBiZW5jaCwgcnVubmVyLCBzdGVlcjogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgICAgICAgICAgbGF5ZXJzPUxBWUVSUywgYWxwaGE6IGZsb2F0ID0gQUxQSEEsIEs6IGZsb2F0ID0gS19QUk9QLAogICAgICAgICAgICAgICAgIHNhZV9yZXBvOiBzdHIgPSBTQUVfUkVQTywgd2VpZ2h0aW5nOiBzdHIgPSAibm9uZSIsCiAgICAgICAgICAgICAgICAgbWlfbWluX2NvdW50OiBpbnQgPSAyKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGJlbmNoLCBydW5uZXIpCiAgICAgICAgc2VsZi5zdGVlciA9IHN0ZWVyIG9yIHR5cGUoc2VsZikuc3RlZXIKICAgICAgICBzZWxmLmxheWVycywgc2VsZi5hbHBoYSwgc2VsZi5LID0gbGlzdChsYXllcnMpLCBhbHBoYSwgSwogICAgICAgIHNlbGYuc2FlX3JlcG8sIHNlbGYud2VpZ2h0aW5nID0gc2FlX3JlcG8sIHdlaWdodGluZwogICAgICAgIHNlbGYubWlfbWluX2NvdW50ID0gbWlfbWluX2NvdW50CiAgICAgICAgc2VsZi5zYWVzOiBkaWN0W2ludCwgX1NBRV0gPSB7fQogICAgICAgIHNlbGYuX2hvb2tzOiBsaXN0ID0gW10KICAgICAgICBzZWxmLl96OiBkaWN0W2ludCwgdHVwbGVdID0ge30gICAjIGxheWVyIC0+IChzZWxfaWR4LCB6X2Rlc2lyZWQsIHpfdW5kZXNpcmVkKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBoaWRkZW4g7IiY7KeRCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIF9oaWRkZW4oc2VsZiwgcmlkczogbnAubmRhcnJheSkgLT4gZGljdFtpbnQsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgIiIi66eI7KeA66eJIOyeheugpSDthqDtgbAoPSDssqsg64u1IO2GoO2BsOydhCDrp4zrk5zripQg7JyE7LmYKeydmCDsuLXrs4QgaGlkZGVuIHN0YXRlLiIiIgogICAgICAgIHIgPSBzZWxmLnJ1bm5lcgogICAgICAgIG91dCA9IHtMOiBbXSBmb3IgTCBpbiBzZWxmLmxheWVyc30KICAgICAgICBmb3IgcyBpbiByYW5nZSgwLCBsZW4ocmlkcyksIDE2KToKICAgICAgICAgICAgYiA9IHJpZHNbczpzICsgMTZdCiAgICAgICAgICAgIGVuYyA9IHIudG9rKHIucHJvbXB0c1tiXS50b2xpc3QoKSwgcmV0dXJuX3RlbnNvcnM9InB0IiwgcGFkZGluZz1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpLnRvKHIubW9kZWwuZGV2aWNlKQogICAgICAgICAgICBocyA9IHIubW9kZWwoKiplbmMsIG91dHB1dF9oaWRkZW5fc3RhdGVzPVRydWUpLmhpZGRlbl9zdGF0ZXMKICAgICAgICAgICAgZm9yIEwgaW4gc2VsZi5sYXllcnM6CiAgICAgICAgICAgICAgICAjIGhpZGRlbl9zdGF0ZXNbTCsxXSA9IEwg67KI7Ke4IGRlY29kZXIgbGF5ZXIg7J2YIOy2nOugpSAoc3BhcnNpZnkg7J2YIGxheWVycy5MIOqzvCDrj5nsnbwpCiAgICAgICAgICAgICAgICBvdXRbTF0uYXBwZW5kKGhzW0wgKyAxXVs6LCAtMSwgOl0uZmxvYXQoKS5jcHUoKSkgICAjIGxlZnQgcGFkZGluZyDihpIgLTEg7J20IOuniOyngOuniQogICAgICAgIHJldHVybiB7TDogdG9yY2guY2F0KHYpIGZvciBMLCB2IGluIG91dC5pdGVtcygpfQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSDrs7TsoJUgKGZvbGQg64K067aAKQoKICAgIGRlZiBfY2FsaWJyYXRlKHNlbGYsIHRyYWluX3JpZHM6IG5wLm5kYXJyYXksIGJhc2U6IHBkLkRhdGFGcmFtZSkgLT4gZGljdDoKICAgICAgICAiIiLCpzQuMX40LjIuIHRyYWluIGZvbGQg7J2YIEPiiJIg7ZaJ66eMIOyCrOyaqe2VnOuLpC4iIiIKICAgICAgICBjbmVnID0gc2VsZi5iZW5jaC5pc19jbmVnCiAgICAgICAgcm93cyA9IG5wLmFycmF5KFtyIGZvciByIGluIHRyYWluX3JpZHMgaWYgY25lZ1tyXV0pCiAgICAgICAgYWNjeCA9IGJhc2UuYWNjX3gudmFsdWVzW3Jvd3NdLmFzdHlwZShib29sKSAgICAgICMgRF9DOiDrrLjrp6Ug64u1IOyDneyEsQogICAgICAgIHJldHggPSBiYXNlLnJldF94LnZhbHVlc1tyb3dzXS5hc3R5cGUoYm9vbCkgICAgICAjIERfTTog7YyM652866mU7Yq466atIOuLtSDsg53shLEKICAgICAgICB1c2UgPSBhY2N4IHwgcmV0eAogICAgICAgIHJvd3MsIHkgPSByb3dzW3VzZV0sIGFjY3hbdXNlXS5hc3R5cGUoaW50KSAgICAgICAjIHk9MSDihpIgQywgeT0wIOKGkiBNCiAgICAgICAgaWYgbWluKCh5ID09IDEpLnN1bSgpLCAoeSA9PSAwKS5zdW0oKSkgPCAzMjoKICAgICAgICAgICAgcHJpbnQoZiJb6rK96rOgXSDrs7TsoJUg7ZGc67O4IOu2gOyhsTogQyB7aW50KCh5PT0xKS5zdW0oKSl9IC8gTSB7aW50KCh5PT0wKS5zdW0oKSl9ICIKICAgICAgICAgICAgICAgICAgIijsm5DrhbzrrLjsnYAgMTI4IOydtOyDgSDqtozsnqUpIikKCiAgICAgICAgSCA9IHNlbGYuX2hpZGRlbihyb3dzKQogICAgICAgIG91dCA9IHt9CiAgICAgICAgZm9yIEwgaW4gc2VsZi5sYXllcnM6CiAgICAgICAgICAgIHNhZSA9IHNlbGYuc2Flc1tMXQogICAgICAgICAgICBpZHgsIHZhbCA9IHNhZS5lbmNvZGUoSFtMXSkKICAgICAgICAgICAgaWR4LCB2YWwgPSBpZHguY3B1KCkubnVtcHkoKSwgdmFsLmNwdSgpLm51bXB5KCkKCiAgICAgICAgICAgICMg7Zmc7ISx65CcIOyggSDsnojripQgZmVhdHVyZSDrp4wg7LaU66aw64ukICgxMzEwNzIg7KCE67aA7JeQIE1JIOulvCDrj4zrpqzrqbQg66ek7JqwIOuKkOumrOuLpCkKICAgICAgICAgICAgZmxhdCA9IGlkeC5yZXNoYXBlKC0xKQogICAgICAgICAgICB1bmlxLCBjbnQgPSBucC51bmlxdWUoZmxhdCwgcmV0dXJuX2NvdW50cz1UcnVlKQogICAgICAgICAgICBjYW5kID0gdW5pcVtjbnQgPj0gc2VsZi5taV9taW5fY291bnRdCiAgICAgICAgICAgIHBvcyA9IHtmOiBqIGZvciBqLCBmIGluIGVudW1lcmF0ZShjYW5kKX0KCiAgICAgICAgICAgIFogPSBucC56ZXJvcygobGVuKHJvd3MpLCBsZW4oY2FuZCkpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICAgICBmb3IgaiBpbiByYW5nZShsZW4ocm93cykpOgogICAgICAgICAgICAgICAgZm9yIGYsIHYgaW4gemlwKGlkeFtqXSwgdmFsW2pdKToKICAgICAgICAgICAgICAgICAgICBwID0gcG9zLmdldChpbnQoZikpCiAgICAgICAgICAgICAgICAgICAgaWYgcCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICAgICAgWltqLCBwXSA9IHYKCiAgICAgICAgICAgIG1pID0gbXV0dWFsX2luZm9fY2xhc3NpZihaLCB5LCBkaXNjcmV0ZV9mZWF0dXJlcz1GYWxzZSwgcmFuZG9tX3N0YXRlPTApCiAgICAgICAgICAgIG9yZGVyID0gbnAuYXJnc29ydCgtbWkpCiAgICAgICAgICAgIHRvdCA9IG1pLnN1bSgpCiAgICAgICAgICAgIGlmIHRvdCA8PSAwOgogICAgICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIkx7TH06IOyDge2YuOygleuztOufieydtCDsoITrtoAgMCDsnbTri6Qg4oCUIOuztOyglSDrjbDsnbTthLDrpbwg7ZmV7J247ZWc64ukIikKICAgICAgICAgICAgayA9IGludChucC5zZWFyY2hzb3J0ZWQobnAuY3Vtc3VtKG1pW29yZGVyXSkgLyB0b3QsIHNlbGYuSykgKyAxKQogICAgICAgICAgICBzZWwgPSBvcmRlcls6a10KCiAgICAgICAgICAgIHpjID0gWlt5ID09IDFdWzosIHNlbF0ubWVhbigwKSAgICAgICAgICAgICAgICMgesyEX0MgKOyEoO2DneuQnCBmZWF0dXJlIOunjCkKICAgICAgICAgICAgem0gPSBaW3kgPT0gMF1bOiwgc2VsXS5tZWFuKDApICAgICAgICAgICAgICAgIyB6zIRfTQogICAgICAgICAgICBpZiBzZWxmLndlaWdodGluZyA9PSAicGFwZXIiOgogICAgICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgid2VpZ2h0aW5nPSdwYXBlcicg64qUIOuvuOq1rO2YhCDigJQg66qo65OIIOyDgeuLqCDso7zshJ0gMuuyiCDssLjsobAiKQoKICAgICAgICAgICAgZGlmZiA9IHpjIC0gem0KICAgICAgICAgICAgel9DID0gbnAud2hlcmUoZGlmZiA+IDAsIHpjLCAwLjApICAgICAgICAgICAgIyDrrLjrp6Ug7IKs7JqpIOyqvSBmdW5jdGlvbmFsIGZlYXR1cmUKICAgICAgICAgICAgel9NID0gbnAud2hlcmUoZGlmZiA8IDAsIHptLCAwLjApICAgICAgICAgICAgIyDtjIzrnbzrqZTtirjrpq0g7IKs7JqpIOyqvQoKICAgICAgICAgICAgZGVzLCB1bmQgPSAoel9NLCB6X0MpIGlmIHNlbGYuc3RlZXIgPT0gIm1lbW9yeSIgZWxzZSAoel9DLCB6X00pCiAgICAgICAgICAgIG91dFtMXSA9ICh0b3JjaC5hc190ZW5zb3IoY2FuZFtzZWxdLCBkdHlwZT10b3JjaC5sb25nKSwKICAgICAgICAgICAgICAgICAgICAgIHRvcmNoLmFzX3RlbnNvcihkZXMsIGR0eXBlPXRvcmNoLmZsb2F0MzIpLAogICAgICAgICAgICAgICAgICAgICAgdG9yY2guYXNfdGVuc29yKHVuZCwgZHR5cGU9dG9yY2guZmxvYXQzMikpCiAgICAgICAgICAgIHByaW50KGYiICBMe0x9OiDtm4Trs7Qge2xlbihjYW5kKX0g4oaSIOyEoO2DnSB7a30gKEs9e3NlbGYuS30pICIKICAgICAgICAgICAgICAgICAgZiJ8IEPsqr0ge2ludCgoZGlmZj4wKS5zdW0oKSl9IC8gTeyqvSB7aW50KChkaWZmPDApLnN1bSgpKX0iKQogICAgICAgIHJldHVybiBvdXQKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0g6rCc7J6FIO2bhQoKICAgIGRlZiBfaW5zdGFsbChzZWxmKSAtPiBOb25lOgogICAgICAgIHIgPSBzZWxmLnJ1bm5lcgogICAgICAgIGZvciBMIGluIHNlbGYubGF5ZXJzOgogICAgICAgICAgICBzYWUsIGRldiA9IHNlbGYuc2Flc1tMXSwgci5tb2RlbC5kZXZpY2UKCiAgICAgICAgICAgIGRlZiBob29rKG1vZCwgYXJncywgb3V0cHV0LCBMPUwsIHNhZT1zYWUpOgogICAgICAgICAgICAgICAgaWYgTCBub3QgaW4gc2VsZi5fejoKICAgICAgICAgICAgICAgICAgICByZXR1cm4gb3V0cHV0CiAgICAgICAgICAgICAgICBoID0gb3V0cHV0WzBdIGlmIGlzaW5zdGFuY2Uob3V0cHV0LCAodHVwbGUsIGxpc3QpKSBlbHNlIG91dHB1dAogICAgICAgICAgICAgICAgaWYgaC5zaGFwZVsxXSA9PSAxOiAgICAgICAgICAjIOybkOuFvOusuDog7J6F66ClIOuniOyngOuniSDsnITsuZjrp4wg7Y647KeRLiBkZWNvZGUg64qUIO2GteqzvAogICAgICAgICAgICAgICAgICAgIHJldHVybiBvdXRwdXQKICAgICAgICAgICAgICAgIHNlbCwgZGVzLCB1bmQgPSBzZWxmLl96W0xdCiAgICAgICAgICAgICAgICBzZWwsIGRlcywgdW5kID0gc2VsLnRvKGRldiksIGRlcy50byhkZXYpLCB1bmQudG8oZGV2KQoKICAgICAgICAgICAgICAgIGxhc3QgPSBoWzosIC0xLCA6XS5mbG9hdCgpICAgICAgICAgICAgICAgICAgICAgICAjIChCLCBkKQogICAgICAgICAgICAgICAgaWR4LCB2YWwgPSBzYWUuZW5jb2RlKGxhc3QpICAgICAgICAgICAgICAgICAgICAgICMgKEIsIGspCiAgICAgICAgICAgICAgICBCID0gbGFzdC5zaGFwZVswXQogICAgICAgICAgICAgICAgeiA9IHRvcmNoLnplcm9zKEIsIGxlbihzZWwpLCBkZXZpY2U9ZGV2KQogICAgICAgICAgICAgICAgbWF0Y2ggPSAoaWR4LnVuc3F1ZWV6ZSgtMSkgPT0gc2VsLnZpZXcoMSwgMSwgLTEpKSAgIyAoQixrLHxzZWx8KQogICAgICAgICAgICAgICAgeiA9ICh2YWwudW5zcXVlZXplKC0xKSAqIG1hdGNoKS5zdW0oMSkgICAgICAgICAgICMg7ISg7YOdIGZlYXR1cmUg7J2YIO2YhOyerCDtmZzshLHqsJIKCiAgICAgICAgICAgICAgICB6X21pbnVzID0gdG9yY2gubWluaW11bSh6LCB1bmQudW5zcXVlZXplKDApKSAgICAgIyBFcS4oMikKICAgICAgICAgICAgICAgIHpfcGx1cyA9IHRvcmNoLmNsYW1wKGRlcy51bnNxdWVlemUoMCkgLSB6LCBtaW49MCkgICMgRXEuKDMpCiAgICAgICAgICAgICAgICBkZWx0YSA9ICh6X3BsdXMgLSB6X21pbnVzKSBAIHNhZS5XX2RlY1tzZWxdLmZsb2F0KCkgICMgRXEuKDQpLCBiaWFzIOyDgeyHhAogICAgICAgICAgICAgICAgaFs6LCAtMSwgOl0gKz0gKHNlbGYuYWxwaGEgKiBkZWx0YSkudG8oaC5kdHlwZSkKICAgICAgICAgICAgICAgIHJldHVybiBvdXRwdXQKCiAgICAgICAgICAgIHNlbGYuX2hvb2tzLmFwcGVuZChyLm1vZGVsLm1vZGVsLmxheWVyc1tMXS5yZWdpc3Rlcl9mb3J3YXJkX2hvb2soaG9vaykpCgogICAgZGVmIF9yZW1vdmUoc2VsZikgLT4gTm9uZToKICAgICAgICBmb3IgaCBpbiBzZWxmLl9ob29rczoKICAgICAgICAgICAgaC5yZW1vdmUoKQogICAgICAgIHNlbGYuX2hvb2tzID0gW10KCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0g7Iuk7ZaJCgogICAgZGVmIHJ1bihzZWxmKToKICAgICAgICByLCBiID0gc2VsZi5ydW5uZXIsIHNlbGYuYmVuY2gKICAgICAgICBkZXYgPSByLm1vZGVsLmRldmljZQogICAgICAgIGJhc2UgPSBsb2FkX2FybSgiYmFzZSIpCgogICAgICAgIHByaW50KGYiW3NwYXJlX3tzZWxmLnN0ZWVyfV0gU0FFIOuhnOuTnCB7c2VsZi5zYWVfcmVwb30iKQogICAgICAgIGZvciBMIGluIHNlbGYubGF5ZXJzOgogICAgICAgICAgICBzZWxmLnNhZXNbTF0gPSBfU0FFKHNlbGYuc2FlX3JlcG8sIEwsIGRldikKICAgICAgICBwcmludCgiICBob29rcG9pbnQ6Iiwge0w6IHMuaG9va3BvaW50IGZvciBMLCBzIGluIHNlbGYuc2Flcy5pdGVtcygpfSkKCiAgICAgICAgIyDsnqzqtazshLEg7ZKI7KeIIO2ZleyduC4gaG9va3BvaW50IOuCmCDrsLHrs7jsnbQg7Ja06riL64KY66m0IOyXrOq4sOyEnCDqsbjrprDri6QuCiAgICAgICAgcHJvYmUgPSBzZWxmLl9oaWRkZW4obnAuYXJhbmdlKDY0KSkKICAgICAgICBmb3IgTCBpbiBzZWxmLmxheWVyczoKICAgICAgICAgICAgZiA9IHNlbGYuc2Flc1tMXS5mdnUocHJvYmVbTF0pCiAgICAgICAgICAgIHByaW50KGYiICBMe0x9IEZWVSB7ZjouM2Z9IikKICAgICAgICAgICAgaWYgZiA+IEZWVV9NQVg6CiAgICAgICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KAogICAgICAgICAgICAgICAgICAgIGYiTHtMfSBGVlUge2Y6LjNmfSA+IHtGVlVfTUFYfS4gU0FFIOqwgCDsnbQg7KeA7KCQ7J2YIO2ZnOyEseydhCDshKTrqoXtlZjsp4Ag66q77ZWc64ukLlxuIgogICAgICAgICAgICAgICAgICAgICIgIC0gaG9va3BvaW50IOq3nOyVvShsYXllciDstpzroKUgdnMg7J6F66ClKeydtCDri6Trpbjsp4BcbiIKICAgICAgICAgICAgICAgICAgICAiICAtIFNBRSDqsIAgYmFzZSDrqqjrjbgg7ZWZ7Iq167O47J24642wIEluc3RydWN0IOyXkCDsoIHsmqnrj7wg6rS066as6rCAIO2BsOyngFxuIgogICAgICAgICAgICAgICAgICAgICLtmZXsnbjtlZzri6QuIOydtCDsg4Htg5zroZwg64+M66awIOyImOy5mOuKlCBTcEFSRSDsnqztmITsnbTrnbwg7ZWgIOyImCDsl4bri6QuIikKCiAgICAgICAgc2VsZi5faW5zdGFsbCgpCiAgICAgICAgdHJ5OgogICAgICAgICAgICByb3dzID0gW10KICAgICAgICAgICAgZm9yIGsgaW4gcmFuZ2UoNSk6CiAgICAgICAgICAgICAgICB0ciA9IG5wLndoZXJlKGIuZm9sZCAhPSBrKVswXQogICAgICAgICAgICAgICAgdGUgPSBucC53aGVyZShiLmZvbGQgPT0gaylbMF0KICAgICAgICAgICAgICAgIHByaW50KGYiW2ZvbGQge2t9XSDrs7TsoJUge2xlbih0cil97ZaJIOKGkiDsoIHsmqkge2xlbih0ZSl97ZaJIikKICAgICAgICAgICAgICAgIHNlbGYuX3ogPSBzZWxmLl9jYWxpYnJhdGUodHIsIGJhc2UpCiAgICAgICAgICAgICAgICBmb3IgcyBpbiByYW5nZSgwLCBsZW4odGUpLCAxNik6CiAgICAgICAgICAgICAgICAgICAgbyA9IHIuZ2VuZXJhdGVfcm93cyh0ZVtzOnMgKyAxNl0pCiAgICAgICAgICAgICAgICAgICAgcm93cyArPSBbZGljdChyaWQ9cmlkLCByYXc9eCwgbl90b2s9biwgZm9sZD1rKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciByaWQsICh4LCBuKSBpbiBvLml0ZW1zKCldCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgc2VsZi5fcmVtb3ZlKCkKCiAgICAgICAgZCA9IG1ha2Vfc2NvcmVyKGIpKHBkLkRhdGFGcmFtZShyb3dzKS5zb3J0X3ZhbHVlcygicmlkIikucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSkKICAgICAgICByZXR1cm4gZC5lbS52YWx1ZXMuYXN0eXBlKGZsb2F0KSwgbGlzdChkLnByZWQudmFsdWVzKQoKCkByZWdpc3Rlcigic3BhcmVfbWVtb3J5IikKY2xhc3MgU3BBUkVNZW1vcnkoX1NwQVJFKToKICAgICIiIu2MjOudvOuplO2KuOumrSDsp4Dsi50g7IKs7JqpIOyqveycvOuhnCDsobDtlqUuIOyasOumrCByZXRhaW4g6rO8IOqwmeydgCDrsKntlqUuIiIiCiAgICBzdGVlciA9ICJtZW1vcnkiCgoKQHJlZ2lzdGVyKCJzcGFyZV9jb250ZXh0IikKY2xhc3MgU3BBUkVDb250ZXh0KF9TcEFSRSk6CiAgICAiIiLrrLjrp6Ug7IKs7JqpIOyqveycvOuhnCDsobDtlqUuIOyasOumrCBhY2NlcHQg7JmAIOqwmeydgCDrsKntlqUo67CY64yAIOq3uSkuIiIiCiAgICBzdGVlciA9ICJjb250ZXh0Igo="
FILES["methods/sota_stubs.py"] = "IiIi7YyA7JuQIOuwsOygleyaqSBTT1RBIOyKpO2FgS4g6rCB7J6QIOunoeydgCDrhbzrrLjsnYQg7JWE656YIO2BtOuemOyKpOyXkCDssYTsmrTri6QuCgpTcEFSRSDripQg6rWs7ZiEIOyZhOujjCDigJQgbWV0aG9kcy9zb3RhX3NwYXJlLnB5IOywuOyhsC4KCuq1rO2YhCDqt5zsuZkKICAxLiBjb21tb24vIOydgCDqs6DsuZjsp4Ag7JWK64qU64ukLgogIDIuIOyDneyEseydgCBjb21tb24uZ2VuZXJhdGUuUnVubmVyIOuhnOunjCDtlZzri6Qo7ZSE66Gs7ZSE7Yq4wrfrsLDsuZjCt0VPUyDqsIAgQTAg7JmAIOqwmeyVhOyVvCDtlZzri6QpLgogIDMuIOyxhOygkOydgCBjb21tb24uc2NvcmluZy5tYWtlX3Njb3JlciDroZzrp4wg7ZWc64ukLgogIDQuIO2VmeyKteydtCDtlYTsmpTtlZjrqbQgYmVuY2guZm9sZCDroZwgNS1mb2xkLCB0ZXN0IGZvbGQg66W8IOuztOyngCDslYrripTri6QuCiAgNS4gcnVuKCkg7J2AIChlbSDrsqHthLAsIHByZWQg66y47J6Q7Je0IOuqqeuhnSkg7J2EIOuwmO2ZmO2VnOuLpC4KCuuwsOyglSDsoJzslYgKICAx7Iic7JyEIFNwQVJFIDog7Jqw66as7JmAIOqwmeydgCDssYTsoJAg67Cp7ZalICsgdHJhaW5pbmctZnJlZSArIOykkeqwhOy4tSDtkZzsg4EgKyBMbGFtYTMtOEIuCiAgICAgICAgICAgICAgICDsl6zquLDshJwg66q7IOydtOq4sOuptCDquLDsl6zqsIAg7ISx66a97ZWY7KeAIOyViuuKlOuLpC4KICAy7Iic7JyEIFNISUZUIDog7ZWZ7Iq1IOq4sOuwmC4g7ISx64qlIOygiOuMgOy5mCDruYTqtZDsmqkuIOyasOumrOuKlCB0cmFpbmluZy1mcmVlIC8g67mE7JqpIOy2leycvOuhnCDtj6zsp4DshZguCiAgM+yInOychCBDT0lFQ0QgLyBDb0NvQSA6IGNvbnRleHQtZmFpdGhmdWwg6rOE7Je0LiDshKTsoJUg7LCo7J20IOyeheymneyaqS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gLmJhc2UgaW1wb3J0IEdlbk1ldGhvZCwgcmVnaXN0ZXIKCgpAcmVnaXN0ZXIoInNoaWZ0IikKY2xhc3MgU0hJRlQoR2VuTWV0aG9kKToKICAgICIiIlNISUZUOiBHYXRlLU1vZHVsYXRlZCBBY3RpdmF0aW9uIFN0ZWVyaW5nIChhclhpdiAyNjA2LjI3Nzg2KS4KICAgIEZGTiDtmZzshLHsl5AgbGVhcm5hYmxlIGdhdGUsIGJhY2tib25lIOqzoOyglSwg7ZWZ7Iq1IO2MjOudvOuvuO2EsCAwLjAxJSDrr7jrp4wuCgogICAg66mU66qoOiDtlZnsirXsnbQg7ZWE7JqU7ZWY66+A66GcIHRyYWluIGZvbGQg66Gc66eMIO2VmeyKte2VnOuLpC4KICAgICAgICAgIOyEseuKpSDsoIjrjIDsuZjroZwg67aZ7Jy866m0IOu2iOumrO2VoCDsiJgg7J6I64ukLiDruYTqtZAg7LaV7J2AIHRyYWluaW5nLWZyZWUgLyDstpTroaAg67mE7JqpLgogICAgIiIiCiAgICBwYXBlciA9ICJTSElGVCAoMjAyNikiCiAgICBmYW1pbHkgPSAiY29uZmxpY3QtcmVzb2x1dGlvbiIKCiAgICBkZWYgcnVuKHNlbGYpOgogICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IKCgpAcmVnaXN0ZXIoImNvaWVjZCIpCmNsYXNzIENPSUVDRChHZW5NZXRob2QpOgogICAgIiIiQ09JRUNEIChZdWFuIGV0IGFsLiwgMjAyNCkuIO2GoO2BsOuzhCDstqnrj4wg7YyQ7KCVIOq4sOuwmCDsoJzslb3soIEg65SU7L2U65SpLiIiIgogICAgcGFwZXIgPSAiQ09JRUNEICgyMDI0KSIKICAgIGZhbWlseSA9ICJjb250ZXh0LWZhaXRoZnVsIgoKICAgIGRlZiBydW4oc2VsZik6CiAgICAgICAgcmFpc2UgTm90SW1wbGVtZW50ZWRFcnJvcgoKCkByZWdpc3RlcigiY29jb2EiKQpjbGFzcyBDb0NvQShHZW5NZXRob2QpOgogICAgIiIiQ29Db0EuIOusuOunpS/tjIzrnbzrqZTtirjrpq0g67aE7Y+sIOqysO2VqSDrlJTsvZTrlKkuIiIiCiAgICBwYXBlciA9ICJDb0NvQSIKICAgIGZhbWlseSA9ICJjb250ZXh0LWZhaXRoZnVsIgoKICAgIGRlZiBydW4oc2VsZik6CiAgICAgICAgcmFpc2UgTm90SW1wbGVtZW50ZWRFcnJvcgo="

PKG_DIR = pathlib.Path(PKG)
for rel, b in FILES.items():
    f = PKG_DIR / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    (f.parent / '__init__.py').touch()
    if f.exists() and not (FORCE_COMMON and rel.startswith('common/')):
        print(f'{rel:32s} 유지 (이미 있음)')
        continue
    f.write_bytes(base64.b64decode(b))
    print(f'{rel:32s} {len(f.read_text().splitlines()):4d}줄  설치')
print('\\n설치 완료 —', PKG)

In [ ]:
import sys, importlib
if PKG not in sys.path:
    sys.path.insert(0, PKG)

def reload_pkg():
    """메서드 파일을 고친 뒤 이 함수를 부르면 다시 읽는다. 세션 재시작 불필요."""
    for mod in list(sys.modules):
        if mod.startswith(("common", "methods")):
            del sys.modules[mod]
    import methods.base, methods.ours_lookup, methods.ours_router  # noqa: F401
    import importlib, pkgutil, methods
    for mi in pkgutil.iter_modules(methods.__path__):
        if mi.name.startswith(("sota_", "ours_")):
            try:
                importlib.import_module(f"methods.{mi.name}")
            except Exception as e:                       # noqa: BLE001
                print(f"  [건너뜀] methods.{mi.name}: {type(e).__name__}: {e}")
    from methods.base import REGISTRY
    return REGISTRY

import numpy as np, pandas as pd
from common.artifacts import load_bench
from common.stats import make_boot, evaluate
REGISTRY = reload_pkg()
print("등록된 메서드:", sorted(REGISTRY))

In [ ]:
## Cell P0 — 내 메서드 파일을 PKG로 복사 (즉시)
import shutil, os
SRC = HK + "src/"
for f in ["ours_packed.py", "ours_ctx.py"]:
    assert os.path.exists(SRC + f), f"없음: {SRC + f} → 바로가기/경로 확인"
    shutil.copy(SRC + f, PKG + "methods/" + f)
REGISTRY = reload_pkg()
print(sorted(k for k in REGISTRY if k.startswith(("packed", "ctx"))))

In [ ]:
for n in ["base", "always_retain", "always_retain_r09"]:
    run(n)

In [ ]:
run("late_router")        # ① routing. 수 분 (L13 hidden 2GB 로드 포함)

In [ ]:
gate = {}
for n, exp in [("always_retain", 0.5997), ("late_router", 0.6078)]:
    got = RES[n]["em"].mean()
    gate[n] = abs(got - exp) <= 5e-3
    print(f"{n}: {got:.4f} vs {exp:.4f} → {'OK' if gate[n] else '불일치'}")

if not all(gate.values()):
    raise SystemExit(
        "재현 실패. 이 순서로 확인한다:\n"
        "  1) ABENCH_A0 의 gen_*.parquet 이 A0 노트북 최신 실행본인가\n"
        "  2) P9/steer_v3/code/p8s_prompt.py md5 가 20959a4c... 인가\n"
        "  3) hidden_v2/p8s_v2_h_out.npy 가 같은 버전인가 (late_router 만 틀릴 때)\n"
        "tolerance 를 늘리거나 기준값을 바꾸지 않는다. 다영에게 알린다.")
print("\n재현 통과 — SOTA 단계로 진행 가능")
save("baselines")

In [ ]:
bench = load_bench()
print(f"{bench.n}행 | 셀 분포 {dict(zip(*np.unique(bench.cell, return_counts=True)))}")

BOOT = make_boot(bench, 2000)
REF  = "base"
RES  = {}

def run(name, **kw):
    """조회형 메서드(기준선) 실행."""
    cls = REGISTRY[name]
    t0 = time.time()
    em, pred = cls(bench, **kw).run()
    em = np.asarray(em, float)
    RES[name] = dict(em=em, pred=list(pred), family=cls.family, paper=cls.paper)
    exp, tag = getattr(cls, "expected_em", None), ""
    if exp is not None:
        tag = f"  | 보고값 {exp:.4f} → {'OK' if abs(em.mean()-exp) <= 5e-3 else '불일치'}"
    print(f"[{name}] EM {em.mean():.4f} | C+ {em[~bench.is_cneg].mean():.4f} "
          f"| C- {em[bench.is_cneg].mean():.4f} ({time.time()-t0:.0f}s){tag}")
    return RES[name]

def run_gen(name, force=False, **kw):
    """생성형 메서드(SOTA) 실행. arm 단위 체크포인트로 세션이 끊겨도 이어진다."""
    f = OUT + f"gen_{name}.parquet"
    if os.path.exists(f) and not force:
        d = pd.read_parquet(f)
        print(f"[{name}] 체크포인트 재사용 EM {d.em.mean():.4f}")
    else:
        t0 = time.time()
        em, pred = REGISTRY[name](bench, runner, **kw).run()
        d = pd.DataFrame(dict(rid=np.arange(bench.n), pred=pred, em=em))
        d.to_parquet(f, index=False)
        print(f"[{name}] EM {d.em.mean():.4f} ({(time.time()-t0)/60:.1f}분)")
    cls = REGISTRY[name]
    RES[name] = dict(em=d.em.values.astype(float), pred=list(d.pred.values),
                     family=cls.family, paper=cls.paper)
    return RES[name]

def save(tag):
    payload = {n: {"em": [int(x) for x in b["em"]], "pred": b["pred"],
                   "family": b["family"], "paper": b["paper"]} for n, b in RES.items()}
    json.dump(payload, open(OUT + f"results_{tag}.json", "w"), ensure_ascii=False)
    print("저장:", OUT + f"results_{tag}.json")

def load(tag):
    d = json.load(open(OUT + f"results_{tag}.json"))
    RES.update({n: {**b, "em": np.array(b["em"], float)} for n, b in d.items()})
    print("복구:", sorted(RES))
    return RES

def table(ref=None):
    ref = ref or REF
    base, refv = RES["base"]["em"], RES[ref]["em"]
    rows = []
    for name, b in RES.items():
        r = evaluate(bench, BOOT, b["em"], base,
                     ref=None if name == ref else refv, name=name)
        r["family"] = b["family"]
        rows.append(r)
    return pd.DataFrame(rows).sort_values("EM", ascending=False)

def show(T, ref=None):
    ref = ref or REF
    mk = lambda x: "O" if x else "X"
    out = [f"| 방법 | 계열 | EM | C+ Δ [하한] | C− Δ | vs {ref} Δ [95% CI] | 개선 | C+ 비열등 | 판정 |",
           "|---|---|---|---|---|---|---|---|---|"]
    for _, r in T.iterrows():
        same = r.policy == ref
        vs = "기준" if same else f"{r.vs_ref*100:+.2f} [{r.vs_ref_lo*100:+.2f}, {r.vs_ref_hi*100:+.2f}]"
        out.append(f"| {r.policy} | {r.family} | {r.EM*100:.2f} "
                   f"| {r.dC_plus*100:+.2f} [{r.dC_plus_lo*100:+.2f}] | {r.dC_minus*100:+.2f} "
                   f"| {vs} | {'—' if same else mk(r.improved)} | {mk(r.safe)} "
                   f"| {'—' if same else mk(r['pass'])} |")
    return "\n".join(out)

print("준비 완료")

In [ ]:
## Cell P1 — packed + w_ctx− (p8s 관문)  A100 약 12분 / L4 약 30분
REGISTRY = reload_pkg()
RAW = OUT + "ours_raw/"
for n in ["packed_x1", "packed_online_s0", "packed_online_s1", "packed_online_s2",
          "ctx_always", "ctx_online_s0", "ctx_online_s1", "ctx_online_s2"]:
    run_gen(n, raw_dir=RAW)
save("with_ours")

from methods.ours_packed import packed_rows, packed_gate_e17
from methods.ours_ctx import seed_rows
packed_gate_e17(bench, RAW, "/content/drive/MyDrive/Grad_share/data_share/E17_ffn_patch/gen")
T = pd.concat([table(), packed_rows(bench, BOOT, RES, REF),
               seed_rows(bench, BOOT, RES, REF, "ctx_online", extra=["ctx_always"])])
print(show(T))